# ChefBot — кулінарний AI-асистент на агентній архітектурі

**Виконала:** Команіч Надія Миколаївна
**Варіант:** 1 — ChefBot (кулінарний асистент)

**Мета роботи.** Перенести архітектуру з практичної роботи (мовна модель → інструменти → агент → контекст) у сценарій кулінарного асистента і показати, які продуктові рішення стоять за кожним блоком коду.

**Що вміє ChefBot:**
1. знаходити рецепти у власній перевіреній базі за назвою або наявними інгредієнтами;
2. точно конвертувати кулінарні міри (склянки, ложки, унції, фунти, °F → °C, мл → г з урахуванням густини продукту);
3. підбирати замінники інгредієнтів із поясненням пропорції та впливу на результат;
4. (розділ 10) враховувати алергії родини, перераховувати порції, оцінювати КБЖВ, радити соуси й напої, вести список покупок і перевіряти числа у відповіді.

**Структура блокнота:**
1. Встановлення та налаштування
2. Мовна модель (`ChatOpenAI`)
3. Інструменти (`@tool`)
4. Системний промпт
5. Агент (`create_agent`)
6. Діалог із контекстом (список `messages`)
7. Автоматичне тестування
8. Таблиця тестування
9. Підсумкова продуктова таблиця та висновки
10. ChefBot+ — розширення: профіль родини й фільтр алергенів, порції кодом, КБЖВ, соуси, напої, покупки, валідатор чисел

## 1. Встановлення та налаштування

In [28]:
!pip install -q -U langchain langchain-openai langchain-core pandas

In [29]:
import os, json, re
from getpass import getpass

# Ключ беремо із секретів Colab (🔑 зліва → OPENAI_API_KEY), а якщо їх немає — просимо ввести вручну.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Введіть OPENAI_API_KEY: ")

print("Ключ завантажено:", bool(os.environ.get("OPENAI_API_KEY")))

Ключ завантажено: True


> **Продуктовий коментар.** Ключ не зберігається в коді блокнота: блокнот здається викладачу і може потрапити в портфоліо, тож секрет живе окремо (Colab Secrets). Це те саме правило, що й у реальному продукті: конфігурація та секрети відокремлені від логіки.

## 2. Мовна модель

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,   # низька креативність: у кулінарії важливі точні пропорції, а не фантазія
    max_tokens=800,
)

# Швидка перевірка, що модель відповідає
print(llm.invoke("Скажи одним реченням українською: що таке бланшування?").content)

Бланшування — це процес короткочасного бланшування продуктів у киплячій воді або парі для збереження кольору, текстури та поживних речовин перед подальшою обробкою або заморожуванням.


> **Продуктовий коментар.** Обрано `gpt-4o-mini`: для ChefBot модель виконує роль *інтерпретатора запиту, диспетчера інструментів і редактора відповіді*, а не джерела фактів — тож велика дорога модель тут не потрібна. `temperature=0.2` — свідоме рішення: користувач на кухні має отримати однакову відповідь на однакове питання, а не «творчу» пропорцію борошна. Креативність потрібна лише у формулюванні порад, і невеликого значення для цього достатньо.

## 3. Інструменти

Головний принцип: **усе, що має бути точним, робить код, а не модель.**

| Інструмент | Що робить | Чому не модель |
|---|---|---|
| `search_recipes` | шукає рецепти в перевіреній базі за назвою або інгредієнтами | модель може «вигадати» рецепт із неправильними пропорціями |
| `convert_units` | конвертує міри за таблицею коефіцієнтів і густин | модель помиляється в арифметиці та плутає US/UK/метричні склянки |
| `find_substitute` | повертає перевірені замінники з пропорцією | замінник без пропорції — шлях до зіпсованої страви |

In [31]:
from langchain_core.tools import tool

# ---------- Дані: міні-база рецептів (у продукті — БД / API) ----------
RECIPES = [
    {
        "name": "Сирники класичні",
        "tags": ["сніданок", "вегетаріанське"],
        "time_min": 25, "servings": 2,
        "ingredients": {"сир кисломолочний": "400 г", "яйце": "1 шт", "борошно": "3 ст. л.",
                        "цукор": "2 ст. л.", "сіль": "дрібка", "олія": "для смаження"},
        "steps": ["Розім'яти сир виделкою.", "Додати яйце, цукор, сіль, перемішати.",
                  "Всипати борошно, сформувати сирники.", "Смажити на середньому вогні по 3–4 хв з кожного боку."],
    },
    {
        "name": "Омлет з овочами",
        "tags": ["сніданок", "швидко", "без глютену"],
        "time_min": 15, "servings": 1,
        "ingredients": {"яйце": "3 шт", "молоко": "50 мл", "помідор": "1 шт",
                        "перець болгарський": "0.5 шт", "сіль": "за смаком", "вершкове масло": "10 г"},
        "steps": ["Нарізати овочі.", "Збити яйця з молоком і сіллю.",
                  "Обсмажити овочі на маслі 2 хв.", "Залити яєчною сумішшю, готувати під кришкою 5–6 хв."],
    },
    {
        "name": "Борщ український",
        "tags": ["обід", "суп"],
        "time_min": 120, "servings": 6,
        "ingredients": {"буряк": "2 шт", "капуста": "300 г", "картопля": "3 шт", "морква": "1 шт",
                        "цибуля": "1 шт", "томатна паста": "2 ст. л.", "свинина": "500 г",
                        "часник": "2 зубчики", "сіль": "за смаком"},
        "steps": ["Зварити бульйон зі свинини (1–1.5 год).", "Додати нарізану картоплю.",
                  "Окремо потушкувати буряк, моркву, цибулю з томатною пастою.",
                  "Додати капусту, через 5 хв — засмажку.", "Довести до кипіння, додати часник, дати настоятися 20 хв."],
    },
    {
        "name": "Паста з томатним соусом",
        "tags": ["обід", "вечеря", "вегетаріанське", "швидко"],
        "time_min": 20, "servings": 2,
        "ingredients": {"паста": "200 г", "помідор": "3 шт", "часник": "2 зубчики",
                        "оливкова олія": "2 ст. л.", "базилік": "кілька листків", "сіль": "за смаком"},
        "steps": ["Відварити пасту al dente.", "Обсмажити часник на олії 30 с.",
                  "Додати нарізані помідори, тушкувати 8–10 хв.", "Змішати з пастою, додати базилік."],
    },
    {
        "name": "Курячий суп з локшиною",
        "tags": ["обід", "суп"],
        "time_min": 50, "servings": 4,
        "ingredients": {"курка": "400 г", "локшина": "100 г", "картопля": "2 шт",
                        "морква": "1 шт", "цибуля": "1 шт", "сіль": "за смаком"},
        "steps": ["Зварити бульйон із курки (30 хв).", "Додати картоплю й моркву, варити 10 хв.",
                  "Додати локшину, варити ще 5–7 хв.", "Посолити, подавати із зеленню."],
    },
    {
        "name": "Млинці на молоці",
        "tags": ["сніданок", "десерт", "вегетаріанське"],
        "time_min": 40, "servings": 4,
        "ingredients": {"молоко": "500 мл", "яйце": "2 шт", "борошно": "200 г",
                        "цукор": "1 ст. л.", "сіль": "дрібка", "олія": "2 ст. л."},
        "steps": ["Збити яйця з цукром і сіллю.", "Додати половину молока і борошно, вимісити без грудочок.",
                  "Влити решту молока й олію.", "Смажити на розігрітій сковорідці по 1 хв з кожного боку."],
    },
    {
        "name": "Овочеве рагу",
        "tags": ["вечеря", "вегетаріанське", "веганське", "без глютену"],
        "time_min": 40, "servings": 3,
        "ingredients": {"кабачок": "1 шт", "картопля": "3 шт", "морква": "1 шт", "цибуля": "1 шт",
                        "перець болгарський": "1 шт", "помідор": "2 шт", "олія": "2 ст. л.", "сіль": "за смаком"},
        "steps": ["Нарізати овочі кубиками.", "Обсмажити цибулю й моркву.",
                  "Додати картоплю, тушкувати 10 хв.", "Додати решту овочів, тушкувати ще 15–20 хв."],
    },
]

def _norm(s: str) -> str:
    return s.lower().strip()


@tool
def search_recipes(query: str = "", ingredients: list[str] | None = None, tag: str = "") -> str:
    """Шукає рецепти у перевіреній базі ChefBot.

    Використовуй, коли користувач просить рецепт, питає «що приготувати»
    або перелічує продукти, які має.

    Args:
        query: назва страви або її частина (наприклад, "борщ"). Може бути порожнім.
        ingredients: список продуктів, які є у користувача (наприклад, ["яйце", "помідор"]).
        tag: фільтр за категорією: "сніданок", "обід", "вечеря", "суп", "швидко",
             "вегетаріанське", "веганське", "без глютену", "десерт". Може бути порожнім.
    Returns:
        JSON зі списком знайдених рецептів (до 3) або повідомлення, що нічого не знайдено.
    """
    ingredients = [_norm(i) for i in (ingredients or []) if i.strip()]
    results = []
    for r in RECIPES:
        if query and not all(w[:4] in _norm(r["name"]) for w in _norm(query).split()):
            continue
        if tag and _norm(tag) not in r["tags"]:
            continue
        score = 0
        if ingredients:
            names = [_norm(k) for k in r["ingredients"]]
            score = sum(any(i[:3] in n for n in names) for i in ingredients)
            if score == 0:
                continue
        results.append((score, r))

    if not results:
        return json.dumps({"found": 0, "message": "У базі немає рецептів за цим запитом."}, ensure_ascii=False)

    results.sort(key=lambda x: -x[0])
    out = []
    for score, r in results[:3]:
        item = dict(r)
        if ingredients:
            names = [_norm(k) for k in r["ingredients"]]
            item["matched_ingredients"] = score
            item["missing_ingredients"] = [k for k in r["ingredients"]
                                           if not any(i[:3] in _norm(k) for i in ingredients)
                                           and k not in ("сіль", "олія")]
        out.append(item)
    return json.dumps({"found": len(out), "recipes": out}, ensure_ascii=False)

In [32]:
# ---------- Конвертація мір ----------
# Об'єм → мілілітри
VOLUME_ML = {
    "мл": 1, "ml": 1, "л": 1000, "l": 1000,
    "склянка": 250, "склянки": 250, "склянок": 250,          # українська гранчаста ~ 250 мл
    "cup": 240, "cups": 240, "чашка": 240, "чашки": 240,     # американська cup
    "ст. л.": 15, "ст.л.": 15, "столова ложка": 15, "столові ложки": 15, "столових ложок": 15, "tbsp": 15,
    "ч. л.": 5, "ч.л.": 5, "чайна ложка": 5, "чайні ложки": 5, "чайних ложок": 5, "tsp": 5,
    "fl oz": 29.57,
}
# Маса → грами
MASS_G = {"г": 1, "g": 1, "кг": 1000, "kg": 1000, "oz": 28.35, "унція": 28.35, "унції": 28.35,
          "lb": 453.6, "фунт": 453.6, "фунти": 453.6}
# Густина, г/мл — потрібна для переходу «об'єм ↔ маса»
DENSITY = {"вода": 1.0, "молоко": 1.03, "борошно": 0.53, "цукор": 0.85, "цукрова пудра": 0.56,
           "сіль": 1.2, "рис": 0.85, "олія": 0.92, "вершкове масло": 0.91, "мед": 1.42,
           "какао": 0.42, "крохмаль": 0.6, "сметана": 1.0}


def _unit_key(u: str, table: dict) -> str | None:
    u = _norm(u)
    if u in table:
        return u
    if u in VOLUME_ML or u in MASS_G:   # точний збіг з іншою таблицею — не вгадуємо
        return None
    for k in table:
        if len(k) >= 3 and len(u) >= 3 and (u.startswith(k) or k.startswith(u)):
            return k
    return None


@tool
def convert_units(amount: float, from_unit: str, to_unit: str, ingredient: str = "") -> str:
    """Точно конвертує кулінарні міри та температуру.

    Використовуй ЗАВЖДИ, коли треба перевести одиниці: склянки, cups, ложки, мл, л,
    г, кг, oz, lb, а також температуру °F ↔ °C. Не рахуй сам.

    Args:
        amount: кількість (число).
        from_unit: вихідна одиниця ДОСЛІВНО так, як її назвав користувач.
            «склянка» передавай як "склянка" (250 мл), «cup» — як "cup" (240 мл).
            Не перекладай і не підміняй одиниці: це різні міри.
            Інші приклади: "ст. л.", "oz", "F".
        to_unit: цільова одиниця, напр. "г", "мл", "C".
        ingredient: продукт — ОБОВ'ЯЗКОВИЙ при переході між об'ємом і масою (напр. "борошно").
    Returns:
        JSON з результатом або поясненням, чому конвертація неможлива.
    """
    f, t = _norm(from_unit).replace("°", ""), _norm(to_unit).replace("°", "")

    # Температура
    if f in ("f", "фаренгейт") and t in ("c", "цельсій"):
        return json.dumps({"result": round((amount - 32) * 5 / 9), "unit": "°C"}, ensure_ascii=False)
    if f in ("c", "цельсій") and t in ("f", "фаренгейт"):
        return json.dumps({"result": round(amount * 9 / 5 + 32), "unit": "°F"}, ensure_ascii=False)

    fv, tv = _unit_key(f, VOLUME_ML), _unit_key(t, VOLUME_ML)
    fm, tm = _unit_key(f, MASS_G), _unit_key(t, MASS_G)

    if fv and tv:
        res = amount * VOLUME_ML[fv] / VOLUME_ML[tv]
    elif fm and tm:
        res = amount * MASS_G[fm] / MASS_G[tm]
    elif (fv and tm) or (fm and tv):
        ing = _norm(ingredient)
        dens = next((v for k, v in DENSITY.items() if k in ing or (ing and ing in k)), None)
        if dens is None:
            return json.dumps({"error": "unknown_density",
                               "message": f"Для переходу між об'ємом і масою потрібна густина продукту. "
                                          f"Продукт '{ingredient or 'не вказано'}' відсутній у таблиці. "
                                          f"Відомі: {', '.join(DENSITY)}."}, ensure_ascii=False)
        if fv:   # об'єм → маса
            res = amount * VOLUME_ML[fv] * dens / MASS_G[tm]
        else:    # маса → об'єм
            res = amount * MASS_G[fm] / dens / VOLUME_ML[tv]
    else:
        return json.dumps({"error": "unknown_unit",
                           "message": f"Невідома одиниця: '{from_unit}' або '{to_unit}'."}, ensure_ascii=False)

    return json.dumps({"amount": amount, "from": from_unit, "to": to_unit, "ingredient": ingredient,
                       "result": round(res, 1)}, ensure_ascii=False)

In [33]:
# ---------- Замінники інгредієнтів ----------
SUBSTITUTES = {
    "яйце": [
        {"замінник": "1 ст. л. меленого льону + 3 ст. л. води (настояти 10 хв)", "пропорція": "на 1 яйце",
         "підходить": "випічка, оладки", "вплив": "щільніша текстура, не підходить для омлету чи безе"},
        {"замінник": "½ стиглого банана (≈50 г)", "пропорція": "на 1 яйце",
         "підходить": "солодка випічка", "вплив": "додає бананового смаку"},
        {"замінник": "60 г натурального йогурту", "пропорція": "на 1 яйце",
         "підходить": "кекси, млинці", "вплив": "м'якша, вологіша текстура"},
    ],
    "вершкове масло": [
        {"замінник": "рослинна олія", "пропорція": "¾ від маси масла",
         "підходить": "кекси, смаження", "вплив": "менше вершкового смаку; для пісочного тіста не підходить"},
        {"замінник": "кокосова олія", "пропорція": "1:1",
         "підходить": "випічка, веганські страви", "вплив": "легкий кокосовий присмак"},
    ],
    "молоко": [
        {"замінник": "вівсяне / мигдальне молоко", "пропорція": "1:1",
         "підходить": "млинці, каші, випічка", "вплив": "майже не змінює результат"},
        {"замінник": "вода + 1 ст. л. масла на 250 мл", "пропорція": "1:1",
         "підходить": "тісто", "вплив": "менш ніжна текстура"},
    ],
    "сметана": [
        {"замінник": "грецький йогурт", "пропорція": "1:1",
         "підходить": "соуси, борщ, випічка", "вплив": "трохи кисліший смак; у гарячому соусі додавати наприкінці"},
    ],
    "борошно пшеничне": [
        {"замінник": "рисове борошно + 1 ч. л. крохмалю на 100 г", "пропорція": "1:1",
         "підходить": "млинці, сирники (без глютену)", "вплив": "крихкіша текстура"},
        {"замінник": "вівсяне борошно", "пропорція": "1:1.2",
         "підходить": "оладки, печиво", "вплив": "горіховий смак, щільніше тісто"},
    ],
    "цукор": [
        {"замінник": "мед", "пропорція": "¾ від маси цукру, рідини в рецепті на 2 ст. л. менше",
         "підходить": "випічка, напої", "вплив": "швидше рум'яниться — знизити температуру на 10–15 °C"},
    ],
    "вершки": [
        {"замінник": "молоко + вершкове масло (3:1)", "пропорція": "1:1",
         "підходить": "соуси", "вплив": "не збиваються у крем"},
    ],
}
ALIASES = {"яйця": "яйце", "масло": "вершкове масло", "борошно": "борошно пшеничне",
           "пшеничне борошно": "борошно пшеничне", "вершків": "вершки"}


@tool
def find_substitute(ingredient: str, dish: str = "") -> str:
    """Підбирає перевірені замінники інгредієнта з пропорцією та впливом на результат.

    Використовуй, коли користувач каже, що якогось продукту немає, у нього алергія,
    або він хоче веганський / безглютеновий варіант.

    Args:
        ingredient: інгредієнт, який треба замінити (напр. "яйце", "молоко").
        dish: страва, для якої потрібна заміна (необов'язково) — допомагає моделі обрати варіант.
    Returns:
        JSON зі списком замінників або повідомлення, що перевірених замін у базі немає.
    """
    key = _norm(ingredient)
    key = ALIASES.get(key, key)
    match = next((k for k in SUBSTITUTES if k == key or key in k or k in key), None)
    if not match:
        return json.dumps({"found": 0, "ingredient": ingredient,
                           "message": "Перевірених замінників у базі немає."}, ensure_ascii=False)
    return json.dumps({"found": len(SUBSTITUTES[match]), "ingredient": match, "dish": dish,
                       "substitutes": SUBSTITUTES[match]}, ensure_ascii=False)


tools = [search_recipes, convert_units, find_substitute]
print("Інструменти:", [t.name for t in tools])

Інструменти: ['search_recipes', 'convert_units', 'find_substitute']


### 3.1. Модульна перевірка інструментів (без моделі)

In [34]:
# Перевіряємо інструменти окремо від агента: якщо тут помилка — винна не модель, а код.
print(convert_units.invoke({"amount": 2, "from_unit": "cup", "to_unit": "г", "ingredient": "борошно"}))
print(convert_units.invoke({"amount": 350, "from_unit": "F", "to_unit": "C"}))
print(convert_units.invoke({"amount": 2, "from_unit": "склянка", "to_unit": "г", "ingredient": "борошно"}))  # 265.0 — українська склянка ≠ cup
print(convert_units.invoke({"amount": 3, "from_unit": "ст. л.", "to_unit": "мл"}))
print(convert_units.invoke({"amount": 1, "from_unit": "склянка", "to_unit": "г", "ingredient": "кіноа"}))
print(find_substitute.invoke({"ingredient": "яйця", "dish": "млинці"})[:200], "...")
print(search_recipes.invoke({"ingredients": ["яйце", "помідор"]})[:200], "...")

{"amount": 2.0, "from": "cup", "to": "г", "ingredient": "борошно", "result": 254.4}
{"result": 177, "unit": "°C"}
{"amount": 2.0, "from": "склянка", "to": "г", "ingredient": "борошно", "result": 265.0}
{"amount": 3.0, "from": "ст. л.", "to": "мл", "ingredient": "", "result": 45.0}
{"error": "unknown_density", "message": "Для переходу між об'ємом і масою потрібна густина продукту. Продукт 'кіноа' відсутній у таблиці. Відомі: вода, молоко, борошно, цукор, цукрова пудра, сіль, рис, олія, вершкове масло, мед, какао, крохмаль, сметана."}
{"found": 3, "ingredient": "яйце", "dish": "млинці", "substitutes": [{"замінник": "1 ст. л. меленого льону + 3 ст. л. води (настояти 10 хв)", "пропорція": "на 1 яйце", "підходить": "випічка, оладки",  ...
{"found": 3, "recipes": [{"name": "Омлет з овочами", "tags": ["сніданок", "швидко", "без глютену"], "time_min": 15, "servings": 1, "ingredients": {"яйце": "3 шт", "молоко": "50 мл", "помідор": "1 шт", ...


> **Продуктовий коментар.** Інструменти тестуються *до* підключення моделі. Так відділяється відповідальність: точність цифр — це зона системи, а модель лише вирішує, *коли* і *з якими параметрами* викликати інструмент.
>
> Ключові рішення:
> - **Docstring — це інструкція для моделі.** Саме з нього модель дізнається, коли викликати інструмент, тому в описах прямо написано «використовуй, коли…» і «не рахуй сам».
> - **Конвертація «об'єм ↔ маса» без густини свідомо відмовляє.** Склянка борошна (≈130 г) і склянка цукру (≈210 г) — різні речі. Краще чесне «не знаю густини кіноа», ніж правдоподібна вигадка.
> - **Розрізнено українську склянку (250 мл) і американську cup (240 мл)** — типова пастка у рецептах з англомовних сайтів.
> - **Замінник завжди повертається з пропорцією і впливом на страву**, бо «заміни яйце бананом» без «½ банана на 1 яйце, і омлет так не зробиш» — непридатна порада.
> - **Пошук рецептів повертає `missing_ingredients`** — користувачу важливо одразу бачити, чого бракує, а не лише «рецепт знайдено».

> **Оновлення після рецензії ментора.** У діалозі розділу 6 користувач написав «2 склянки борошна», а модель передала інструменту `from_unit='cup'` (240 мл) і отримала 254.4 г замість 265 г для української склянки. Інструмент розрізняв склянку і cup правильно — одиницю «переклала» сама модель.
>
> Що змінено: у docstring `convert_units` і в системному промпті додано правило «передавай одиницю так, як її назвав користувач»; у модульну перевірку додано виклик зі «склянкою» (265.0 г); у тест-набір — сценарій **T13 «Склянка ≠ cup»**.
>
> Результати запуску нижче — з попередньої версії. Після змін блокнот треба перезапустити (Runtime → Run all), щоб T13 і діалог розділу 6 показали нову поведінку.


## 4. Системний промпт

In [35]:
SYSTEM_PROMPT = """Ти — ChefBot, дружній і точний кулінарний асистент для домашньої кухні.
Відповідай українською, коротко і по ділу, тоном досвідченої господині, яка радо ділиться порадою.

## Твої інструменти та правила їх використання
1. search_recipes — для БУДЬ-ЯКОГО запиту на рецепт або «що приготувати з…».
   - Якщо користувач перелічує продукти, передай їх у параметр ingredients.
   - Подавай лише рецепти, які повернув інструмент. НЕ вигадуй рецептів і пропорцій.
   - Якщо інструмент нічого не знайшов — чесно скажи, що в базі такого рецепта немає,
     і запропонуй найближчу альтернативу з бази або уточнення.
   - Покажи, яких інгредієнтів бракує (поле missing_ingredients).
2. convert_units — для БУДЬ-ЯКОЇ конвертації мір і температури. Ніколи не рахуй сам.
   - Для переходу між об'ємом і масою обов'язково передавай ingredient.
   - Передавай одиницю так, як її назвав користувач: «склянка» — це не «cup». Не перекладай одиниці.
   - Якщо інструмент повернув помилку густини — поясни це і запропонуй виміряти вагою.
3. find_substitute — коли продукту немає, є алергія або потрібен веганський/безглютеновий варіант.
   - Завжди називай пропорцію і як заміна вплине на страву.
   - Якщо перевіреного замінника немає — так і скажи, не вигадуй.

Можна викликати кілька інструментів послідовно (наприклад, знайти рецепт, а потім замінити інгредієнт).

## Контекст діалогу
- Пам'ятай, що користувач сказав раніше: алергії, обмеження, кількість порцій, наявні продукти,
  обрану страву. Якщо користувач пише «а для нього…» чи «а без яєць?» — це про попередню страву.
- Якщо користувач повідомив про алергію — ніколи не пропонуй рецепт із цим продуктом без заміни.

## Межі поведінки
- Ти відповідаєш ЛИШЕ на кулінарні теми: рецепти, міри, замінники, техніки приготування.
  На інші теми ввічливо відмовляй і повертай розмову до кухні.
- Ти НЕ дієтолог і НЕ лікар: не складай лікувальних дієт і не давай медичних порад щодо алергій
  чи хвороб — порадь звернутися до фахівця. Підібрати заміну алергену в рецепті можна.
- Не давай порад, небезпечних для здоров'я (сире м'ясо курки, консервування без стерилізації тощо).
- Якщо запит неоднозначний — постав одне коротке уточнювальне питання.
"""
print(f"Довжина промпту: {len(SYSTEM_PROMPT)} символів")

Довжина промпту: 2183 символів


> **Продуктовий коментар.** Промпт побудований як три шари: **роль** (хто ChefBot і яким тоном говорить), **правила інструментів** (коли що викликати і що робити, якщо інструмент нічого не дав), **межі** (що бот не робить).
>
> Найважливіше рішення — правило *«не вигадуй, якщо інструмент нічого не знайшов»*. Без нього модель охоче «допише» відсутній рецепт, і користувач не відрізнить перевірене від згенерованого. Друге — межа *«не дієтолог і не лікар»*: кулінарний продукт часто отримує питання про здоров'я, і бот має знати, де зупинитися, але при цьому лишатися корисним (заміну алергену в рецепті підібрати може).

## 5. Агент

In [36]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)
print("Агент створений ✓")

Агент створений ✓


In [37]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def used_tools(messages) -> list[str]:
    """Повертає список інструментів, які агент викликав у межах одного ходу."""
    names = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            names += [tc["name"] for tc in m.tool_calls]
    return names


def ask_once(question: str, verbose: bool = True) -> dict:
    """Одиночний запит без історії — для перевірки окремих сценаріїв."""
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    answer = result["messages"][-1].content
    tools_called = used_tools(result["messages"])
    if verbose:
        print(f"👤 {question}\n🔧 Інструменти: {tools_called or '—'}\n🤖 {answer}\n" + "-" * 80)
    return {"answer": answer, "tools": tools_called, "messages": result["messages"]}


_ = ask_once("Як приготувати сирники?")

👤 Як приготувати сирники?
🔧 Інструменти: ['search_recipes']
🤖 Ось рецепт класичних сирників:

### Інгредієнти:
- Сир кисломолочний — 400 г
- Яйце — 1 шт
- Борошно — 3 ст. л.
- Цукор — 2 ст. л.
- Сіль — дрібка
- Олія — для смаження

### Приготування:
1. Розім'яти сир виделкою.
2. Додати яйце, цукор, сіль, перемішати.
3. Всипати борошно, сформувати сирники.
4. Смажити на середньому вогні по 3–4 хв з кожного боку.

Смачного!
--------------------------------------------------------------------------------


> **Продуктовий коментар.** `create_agent` реалізує цикл *модель → виклик інструмента → результат назад у модель → …*, доки модель не сформує фінальну відповідь. Тут модель виконує три функції: **інтерпретує** запит (що хоче людина), **обирає інструмент і параметри**, **формулює** відповідь з даних, які повернув інструмент. Пошук, обчислення і дані — поза моделлю.
>
> Функція `used_tools` додана не для краси: вона робить поведінку агента *видимою*. Без неї неможливо перевірити, чи бот справді порахував через `convert_units`, чи «відповів з голови».

## 6. Діалог із контекстом

In [38]:
class ChefBotChat:
    """Діалог, у якому вся історія зберігається у списку messages і передається агенту щоразу."""

    def __init__(self, max_messages: int = 30):
        self.messages: list = []
        self.max_messages = max_messages   # обмеження, щоб контекст не ріс безкінечно

    def ask(self, text: str, verbose: bool = True) -> str:
        self.messages.append(HumanMessage(content=text))
        result = agent.invoke({"messages": self.messages})
        new_part = result["messages"][len(self.messages):]
        self.messages = result["messages"]          # зберігаємо і виклики інструментів, і відповіді
        if len(self.messages) > self.max_messages:  # обрізаємо найстаріше, починаючи з репліки людини
            cut = self.messages[-self.max_messages:]
            while cut and not isinstance(cut[0], HumanMessage):
                cut = cut[1:]
            self.messages = cut
        answer = self.messages[-1].content
        if verbose:
            print(f"👤 {text}\n🔧 Інструменти: {used_tools(new_part) or '—'}\n🤖 {answer}\n" + "-" * 80)
        return answer

    def reset(self):
        self.messages = []

In [39]:
chat = ChefBotChat()

chat.ask("Привіт! У мене алергія на яйця. Хочу на сніданок млинці.")
chat.ask("А скільки це буде грамів, якщо мені треба 2 склянки борошна?")
chat.ask("А якщо готувати на 2 порції, а не на 4 — скільки тоді молока?")
chat.ask("Дякую! А що можна приготувати на вечерю з кабачка та картоплі?")

👤 Привіт! У мене алергія на яйця. Хочу на сніданок млинці.
🔧 Інструменти: ['find_substitute', 'search_recipes', 'search_recipes']
🤖 На жаль, у базі немає рецептів млинців без яєць. Але можу запропонувати загальний рецепт млинців, до якого можна додати замінники для яєць:

### Млинці на молоці (для 4 порцій)
**Інгредієнти:**
- 500 мл молока
- 200 г борошна
- 1 ст. л. цукру
- дрібка солі
- 2 ст. л. олії
- **Замінник для яєць:** 
  - 1 ст. л. меленого льону + 3 ст. л. води (настояти 10 хв) 
  - або ½ стиглого банана (≈50 г) 
  - або 60 г натурального йогурту

**Приготування:**
1. Змішайте замінник з цукром і сіллю.
2. Додайте половину молока і борошно, вимісити без грудочок.
3. Влити решту молока й олію.
4. Смажити на розігрітій сковідці по 1 хв з кожного боку.

Спробуйте один із замінників для яєць, щоб отримати смачні млинці!
--------------------------------------------------------------------------------
👤 А скільки це буде грамів, якщо мені треба 2 склянки борошна?
🔧 Інструменти: ['co

'Можна приготувати **овочеве рагу**. Ось рецепт:\n\n### Овочеве рагу (на 3 порції)\n**Інгредієнти:**\n- 1 кабачок\n- 3 картоплі\n- 1 морква (бракує)\n- 1 цибуля (бракує)\n- 1 болгарський перець (бракує)\n- 2 помідори (бракує)\n- 2 ст. л. олії\n- сіль за смаком\n\n**Приготування:**\n1. Нарізати овочі кубиками.\n2. Обсмажити цибулю й моркву.\n3. Додати картоплю, тушкувати 10 хв.\n4. Додати решту овочів, тушкувати ще 15–20 хв.\n\nДля приготування вам знадобляться морква, цибуля, болгарський перець і помідори. Якщо чогось не вистачає, можу допомогти знайти заміну!'

In [40]:
# Що реально лежить в історії — видно, як система «пам'ятає»
for i, m in enumerate(chat.messages):
    role = type(m).__name__.replace("Message", "")
    content = (m.content or str(getattr(m, "tool_calls", "")))[:90].replace("\n", " ")
    print(f"{i:>2}. {role:<6} | {content}")

 0. Human  | Привіт! У мене алергія на яйця. Хочу на сніданок млинці.
 1. AI     | [{'name': 'find_substitute', 'args': {'ingredient': 'яйце', 'dish': 'млинці'}, 'id': 'call
 2. Tool   | {"found": 3, "ingredient": "яйце", "dish": "млинці", "substitutes": [{"замінник": "1 ст. л
 3. AI     | [{'name': 'search_recipes', 'args': {'query': 'млинці', 'ingredients': ['мелений льон', 'с
 4. Tool   | {"found": 0, "message": "У базі немає рецептів за цим запитом."}
 5. AI     | [{'name': 'search_recipes', 'args': {'query': 'млинці', 'tag': 'сніданок'}, 'id': 'call_BF
 6. Tool   | {"found": 1, "recipes": [{"name": "Млинці на молоці", "tags": ["сніданок", "десерт", "веге
 7. AI     | На жаль, у базі немає рецептів млинців без яєць. Але можу запропонувати загальний рецепт м
 8. Human  | А скільки це буде грамів, якщо мені треба 2 склянки борошна?
 9. AI     | [{'name': 'convert_units', 'args': {'amount': 2, 'from_unit': 'склянка', 'to_unit': 'г', '
10. Tool   | {"amount": 2.0, "from": "склянка", "t

> **Продуктовий коментар.** Контекст зберігається як список `messages`, і агент щоразу отримує його повністю. Тому третє питання («а на 2 порції — скільки молока?») бот розуміє без повторення назви страви, а алергію з першої репліки враховує у всіх наступних відповідях.
>
> У список потрапляють не лише репліки людини і бота, а й виклики інструментів з їхніми результатами: модель бачить, *які саме цифри* вона вже отримала, і не перераховує їх.
>
> Обмеження `max_messages` — компроміс між пам'яттю і вартістю: довгий контекст дорожчий і повільніший. Обрізання починається з репліки людини, щоб не залишити «осиротілий» результат інструмента без запиту, який його викликав. Ціна цього рішення — у дуже довгій розмові бот може «забути» алергію з початку; у реальному продукті такі факти мають зберігатися окремо, у профілі користувача (див. слабкі місця).

## 7. Автоматичне тестування

In [41]:
TEST_CASES = [
    # (id, категорія, запит, очікувані інструменти, що перевіряємо)
    ("T1", "Пошук рецепта", "Дай рецепт борщу", ["search_recipes"],
     "рецепт із бази, без вигаданих пропорцій"),
    ("T2", "Пошук за інгредієнтами", "У мене є яйця, помідор і перець. Що приготувати?", ["search_recipes"],
     "передає інгредієнти, показує, чого бракує"),
    ("T3", "Конвертація об'єм→маса", "Скільки грамів у 2 cups борошна?", ["convert_units"],
     "≈254 г, рахує інструмент"),
    ("T4", "Конвертація температури", "Духовка 350 F — це скільки за Цельсієм?", ["convert_units"],
     "≈177 °C"),
    ("T5", "Невідома густина", "Скільки грамів у склянці кіноа?", ["convert_units"],
     "чесно пояснює, що густини немає"),
    ("T6", "Замінник", "Чим замінити вершкове масло у кексах?", ["find_substitute"],
     "замінник + пропорція + вплив"),
    ("T7", "Ланцюжок інструментів", "Хочу сирники, але без яєць. Що робити?", ["search_recipes", "find_substitute"],
     "знаходить рецепт і підбирає заміну яйця"),
    ("T8", "Рецепта немає в базі", "Дай рецепт тайського супу том ям", ["search_recipes"],
     "не вигадує рецепт, чесно каже, що в базі немає"),
    ("T9", "Поза межами (тема)", "Яка завтра буде погода у Львові?", [],
     "ввічлива відмова, повернення до кухні"),
    ("T10", "Поза межами (медицина)", "Склади мені лікувальну дієту від гастриту на тиждень", [],
     "не дає медичних порад, радить фахівця"),
    ("T11", "Неоднозначний запит", "Скільки це в грамах?", [],
     "ставить уточнювальне питання"),
    ("T13", "Склянка ≠ cup", "Скільки грамів у 2 склянках борошна?", ["convert_units"],
     "≈265 г: інструменту передано «склянка», а не «cup»"),
]

import pandas as pd
rows, raw = [], {}
for tid, cat, q, expected, check in TEST_CASES:
    print(f"=== {tid} · {cat} ===")
    r = ask_once(q)
    raw[tid] = r
    called = r["tools"]
    tools_ok = set(expected) <= set(called) if expected else len(called) == 0
    rows.append({"ID": tid, "Сценарій": cat, "Запит": q,
                 "Очікувані інструменти": ", ".join(expected) or "—",
                 "Фактичні інструменти": ", ".join(called) or "—",
                 "Інструменти ✓": "✅" if tools_ok else "⚠️",
                 "Що перевіряємо": check,
                 "Відповідь (початок)": r["answer"][:160].replace("\n", " ") + "…"})

=== T1 · Пошук рецепта ===
👤 Дай рецепт борщу
🔧 Інструменти: ['search_recipes']
🤖 Ось рецепт українського борщу:

### Інгредієнти (на 6 порцій):
- Буряк — 2 шт
- Капуста — 300 г
- Картопля — 3 шт
- Морква — 1 шт
- Цибуля — 1 шт
- Томатна паста — 2 ст. л.
- Свинина — 500 г
- Часник — 2 зубчики
- Сіль — за смаком

### Приготування:
1. Зварити бульйон зі свинини (1–1.5 год).
2. Додати нарізану картоплю.
3. Окремо потушкувати буряк, моркву, цибулю з томатною пастою.
4. Додати капусту, через 5 хв — засмажку.
5. Довести до кипіння, додати часник, дати настоятися 20 хв.

Смачного!
--------------------------------------------------------------------------------
=== T2 · Пошук за інгредієнтами ===
👤 У мене є яйця, помідор і перець. Що приготувати?
🔧 Інструменти: ['search_recipes']
🤖 Ось кілька ідей, що можна приготувати з яєць, помідора і перцю:

1. **Омлет з овочами** (сніданок, швидко, без глютену)
   - **Інгредієнти**: 3 яйця, 50 мл молока, 1 помідор, 0.5 болгарського перцю, сіль за смаком, 

In [42]:
# Перевірка контексту як окремий автоматичний тест
print("=== T12 · Контекст (алергія + уточнення порцій) ===")
ctx = ChefBotChat()
a1 = ctx.ask("У мене алергія на молоко. Хочу млинці.")
a2 = ctx.ask("А скільки потрібно борошна, якщо робити половину порції?")
ctx_tools = used_tools(ctx.messages)
mentions_sub = any(w in (a1 + a2).lower() for w in ["вівсян", "мигдал", "рослинн", "замін"])
rows.append({"ID": "T12", "Сценарій": "Контекст діалогу",
             "Запит": "алергія на молоко → млинці → «а скільки борошна на половину?»",
             "Очікувані інструменти": "search_recipes, find_substitute",
             "Фактичні інструменти": ", ".join(ctx_tools) or "—",
             "Інструменти ✓": "✅" if {"search_recipes", "find_substitute"} <= set(ctx_tools) else "⚠️",
             "Що перевіряємо": "пам'ятає страву та алергію; 100 г борошна; заміна молока",
             "Відповідь (початок)": a2[:160].replace("\n", " ") + "…"})
print("Заміна молока згадана:", mentions_sub)

=== T12 · Контекст (алергія + уточнення порцій) ===
👤 У мене алергія на молоко. Хочу млинці.
🔧 Інструменти: ['find_substitute', 'search_recipes']
🤖 Для приготування млинців без молока, ви можете використовувати вівсяне або мигдальне молоко в пропорції 1:1 замість звичайного молока. Це майже не змінить смак.

Ось рецепт млинців, в якому потрібно замінити молоко:

### Млинці на молоці (з заміною)
**Інгредієнти:**
- Вівсяне або мигдальне молоко — 500 мл (замість звичайного молока)
- Яйце — 2 шт
- Борошно — 200 г
- Цукор — 1 ст. л.
- Сіль — дрібка
- Олія — 2 ст. л.

**Приготування:**
1. Збити яйця з цукром і сіллю.
2. Додати половину вівсяного або мигдального молока і борошно, вимісити без грудочок.
3. Влити решту молока й олію.
4. Смажити на розігрітій сковорідці по 1 хв з кожного боку.

Смачного! Якщо вам потрібні ще поради, звертайтеся!
--------------------------------------------------------------------------------
👤 А скільки потрібно борошна, якщо робити половину порції?
🔧 Інструмент

In [43]:
pd.set_option("display.max_colwidth", 120)
results_df = pd.DataFrame(rows)
display(results_df)

passed = (results_df["Інструменти ✓"] == "✅").sum()
print(f"\nКоректний вибір інструментів: {passed} з {len(results_df)}")

,ID,Сценарій,Запит,Очікувані інструменти,Фактичні інструменти,Інструменти ✓,Що перевіряємо,Відповідь (початок)
0,T1,Пошук рецепта,Дай рецепт борщу,search_recipes,search_recipes,✅,"рецепт із бази, без вигаданих пропорцій",Ось рецепт українського борщу: ### Інгредієнти (на 6 порцій): - Буряк — 2 шт - Капуста — 300 г - Картопля — 3 шт - ...
1,T2,Пошук за інгредієнтами,"У мене є яйця, помідор і перець. Що приготувати?",search_recipes,search_recipes,✅,"передає інгредієнти, показує, чого бракує","Ось кілька ідей, що можна приготувати з яєць, помідора і перцю: 1. **Омлет з овочами** (сніданок, швидко, без глюте..."
2,T3,Конвертація об'єм→маса,Скільки грамів у 2 cups борошна?,convert_units,convert_units,✅,"≈254 г, рахує інструмент",У 2 cups борошна приблизно 254.4 грамів.…
3,T4,Конвертація температури,Духовка 350 F — це скільки за Цельсієм?,convert_units,convert_units,✅,≈177 °C,Духовка 350°F — це приблизно 177°C.…
4,T5,Невідома густина,Скільки грамів у склянці кіноа?,convert_units,convert_units,✅,"чесно пояснює, що густини немає","На жаль, я не можу точно конвертувати об'єм кіноа в грами через відсутність даних про його густину. Рекомендую вимір..."
5,T6,Замінник,Чим замінити вершкове масло у кексах?,find_substitute,find_substitute,✅,замінник + пропорція + вплив,Можеш замінити вершкове масло у кексах на: 1. **Рослинна олія** — використай ¾ від маси масла. Це підходить для кек...
6,T7,Ланцюжок інструментів,"Хочу сирники, але без яєць. Що робити?","search_recipes, find_substitute","find_substitute, search_recipes",✅,знаходить рецепт і підбирає заміну яйця,"Для приготування сирників без яєць, можна використати один з таких замінників: 1. **1 ст. л. меленого льону + 3 ст...."
7,T8,Рецепта немає в базі,Дай рецепт тайського супу том ям,search_recipes,search_recipes,✅,"не вигадує рецепт, чесно каже, що в базі немає","На жаль, у базі немає рецепту тайського супу том ям. Можу запропонувати приготувати інший суп або уточнити, які інгр..."
8,T9,Поза межами (тема),Яка завтра буде погода у Львові?,—,—,✅,"ввічлива відмова, повернення до кухні","На жаль, я не можу допомогти з прогнозом погоди. Але можу підказати, що приготувати на сніданок або обід! Чим можу д..."
9,T10,Поза межами (медицина),Склади мені лікувальну дієту від гастриту на тиждень,—,—,✅,"не дає медичних порад, радить фахівця","На жаль, я не можу допомогти з лікувальними дієтами. Рекомендую звернутися до лікаря або дієтолога. Якщо вам потрібе..."



Коректний вибір інструментів: 13 з 13


In [44]:
# Перевірка точності: числа в відповідях збігаються з розрахунком інструмента
checks = {
    "T3": ("254", "2 cups борошна ≈ 254 г"),
    "T4": ("177", "350 °F ≈ 177 °C"),
    "T13": ("265", "2 склянки борошна ≈ 265 г (а не 254.4 г для cup)"),
}
for tid, (num, label) in checks.items():
    ok = num in raw[tid]["answer"].replace(" ", "")
    print(f"{tid}: {label} → {'✅ число у відповіді' if ok else '⚠️ перевірити вручну'}")

# Перевірка «не вигадує»: у T8 не має бути переліку інгредієнтів зі вигаданими грамами
t8 = raw["T8"]["answer"].lower()
print("T8:", "✅ чесно повідомив про відсутність" if any(w in t8 for w in ["немає", "не знайш", "відсутн"])
      else "⚠️ можливо, вигадав рецепт — перевірити")

T3: 2 cups борошна ≈ 254 г → ✅ число у відповіді
T4: 350 °F ≈ 177 °C → ✅ число у відповіді
T13: 2 склянки борошна ≈ 265 г (а не 254.4 г для cup) → ✅ число у відповіді
T8: ✅ чесно повідомив про відсутність


> **Продуктовий коментар.** Тест-набір побудований не з «зручних» запитів, а за категоріями ризику: основні функції (T1–T4, T6), ланцюжок інструментів (T7), *чесність при відсутності даних* (T5, T8), межі продукту (T9, T10), неоднозначність (T11) і пам'ять (T12).
>
> Перевіряються дві речі окремо: **чи правильний інструмент обрано** (автоматично, за `tool_calls`) і **чи правильний зміст** (числа в T3/T4, відсутність вигадки в T8). Для LLM-систем «зелений» перший пункт не гарантує другого — модель може викликати інструмент і все одно перекрутити результат, тож ключові числа звіряються з розрахунком.
>
> Відповіді моделі недетерміновані, тому результати фіксуються саме тим запуском, який подається на перевірку.

## 8. Таблиця тестування

Заповнено за результатами запуску (автоматична таблиця `results_df` вище). Оцінка: ✅ — критерій виконано, ⚠️ — виконано частково, ❌ — не виконано.

| ID | Сценарій | Запит | Очікуваний інструмент | Критерій успіху | Фактичний результат | Оцінка |
|---|---|---|---|---|---|---|
| T1 | Пошук рецепта | Дай рецепт борщу | `search_recipes` | Рецепт із бази без вигадок | Викликано `search_recipes`, рецепт подано точно за базою (6 порцій, усі кроки) | ✅ |
| T2 | Пошук за інгредієнтами | Є яйця, помідор, перець | `search_recipes` | Омлет з овочами; видно, чого бракує | Омлет — перший у списку, для кожної страви вказано «бракує» | ✅ |
| T3 | Об'єм → маса | 2 cups борошна в грамах | `convert_units` | ≈254 г | 254.4 г — число з інструмента | ✅ |
| T4 | Температура | 350 °F → °C | `convert_units` | ≈177 °C | 177 °C | ✅ |
| T5 | Невідома густина | Склянка кіноа в грамах | `convert_units` | Чесне пояснення + порада зважити | Пояснив, порадив зважити, але **додав «зазвичай ≈170 г» від себе** — цифра не з інструмента | ⚠️ |
| T6 | Замінник | Замінити масло в кексах | `find_substitute` | Замінник + пропорція + вплив | Два замінники, з пропорцією і впливом на смак | ✅ |
| T7 | Ланцюжок | Сирники без яєць | `search_recipes` + `find_substitute` | Рецепт + заміна яйця | Викликано **лише** `find_substitute`; рецепт не показано; серед варіантів — банан «для солодкої випічки», що для сирників не відфільтровано | ⚠️ |
| T8 | Немає в базі | Том ям | `search_recipes` | Не вигадує, пропонує альтернативу з бази | Чесно сказав, що рецепта немає, але **запропонував «грибний суп», якого в базі немає** | ⚠️ |
| T9 | Поза темою | Погода у Львові | — | Ввічлива відмова, повернення до кухні | Відмовив і запропонував кулінарну допомогу | ✅ |
| T10 | Медицина | Лікувальна дієта | — | Не дає медичних порад, радить фахівця | Від дієти відмовився, але **не порадив звернутися до лікаря/дієтолога** | ⚠️ |
| T11 | Неоднозначність | Скільки це в грамах? | — | Уточнювальне питання | Попросив уточнити кількість і одиницю | ✅ |
| T13 | Склянка ≠ cup | 2 склянки борошна в грамах | `convert_units` | ≈265 г, одиниця «склянка» | *заповнити після перезапуску* | |
| T12 | Контекст | Алергія на молоко → млинці → половина порції | `search_recipes` + `find_substitute` | Пам'ятає страву й алергію; 100 г борошна | Молоко замінено на рослинне прямо в рецепті; на уточнення — 100 г борошна (контекст утримано) | ✅ |

**Підсумок:** 7 сценаріїв ✅, 5 — ⚠️, ❌ немає. Інструменти обрано правильно в 11 з 12 випадків; усі числа, які пройшли через інструменти, точні.

### Аналіз відхилень

**1. Модель «доповнює» відповідь власними знаннями поверх інструмента (T5, T8).** Інструмент чесно повернув «не знаю», модель це передала — і тут же додала від себе цифру (170 г) чи страву («грибний суп»). Для користувача ці слова виглядають так само достовірно, як і перевірені дані. Це саме той ризик, через який рахунок і рецепти винесено в інструменти: правило «не вигадуй» у промпті зменшує його, але не усуває.

**2. Агент не завжди будує ланцюжок інструментів (T7, діалог у розділі 6).** На «сирники без яєць» бот підібрав замінник, але не відкрив рецепт. У діалозі з розділу 6 на «млинці, алергія на яйця» він викликав обидва інструменти, але сказав, що «рецептів млинців без яєць немає», замість того щоб показати наявний рецепт із заміною. Через це на наступне питання («на 2 порції — скільки молока?») бот не мав рецепта в контексті й **порахував від вигаданої бази** (½ склянки ≈ 124 г), хоча за рецептом на 2 порції потрібно 250 мл. Це найсерйозніша помилка запуску: технічно інструмент викликано, але вихідні дані для нього модель взяла «з голови».

**3. Межі дотримано, але без «м'якого виходу» (T10).** Медичну тему бот відсік правильно, проте не скерував людину до фахівця, хоча промпт цього вимагає.

**Що змінити в наступній ітерації:**
- у промпт додати: «якщо інструмент не дав даних — не додавай власних цифр і страв; пропонуй лише те, що є в базі»;
- додати правило: «якщо користувач просить страву з обмеженням — спершу `search_recipes`, потім `find_substitute`, і покажи рецепт уже з заміною»;
- додати інструмент `scale_recipe(recipe, servings)`, щоб масштабування порцій рахував код, а не модель;
- у T10 явно вимагати фразу-скерування до лікаря/дієтолога.

In [45]:
# Автоматично формуємо заповнену таблицю тестування у markdown — її можна вставити у клітинку вище
from IPython.display import Markdown
lines = ["| ID | Сценарій | Інструменти (очік. / факт.) | Оцінка | Відповідь (початок) |", "|---|---|---|---|---|"]
for r in rows:
    lines.append(f"| {r['ID']} | {r['Сценарій']} | {r['Очікувані інструменти']} / {r['Фактичні інструменти']} "
                 f"| {r['Інструменти ✓']} | {r['Відповідь (початок)'].replace('|', '/')} |")
display(Markdown("\n".join(lines)))

| ID | Сценарій | Інструменти (очік. / факт.) | Оцінка | Відповідь (початок) |
|---|---|---|---|---|
| T1 | Пошук рецепта | search_recipes / search_recipes | ✅ | Ось рецепт українського борщу:  ### Інгредієнти (на 6 порцій): - Буряк — 2 шт - Капуста — 300 г - Картопля — 3 шт - Морква — 1 шт - Цибуля — 1 шт - Томатна паст… |
| T2 | Пошук за інгредієнтами | search_recipes / search_recipes | ✅ | Ось кілька ідей, що можна приготувати з яєць, помідора і перцю:  1. **Омлет з овочами** (сніданок, швидко, без глютену)    - **Інгредієнти**: 3 яйця, 50 мл моло… |
| T3 | Конвертація об'єм→маса | convert_units / convert_units | ✅ | У 2 cups борошна приблизно 254.4 грамів.… |
| T4 | Конвертація температури | convert_units / convert_units | ✅ | Духовка 350°F — це приблизно 177°C.… |
| T5 | Невідома густина | convert_units / convert_units | ✅ | На жаль, я не можу точно конвертувати об'єм кіноа в грами через відсутність даних про його густину. Рекомендую виміряти кіноа вагою, якщо це можливо. Зазвичай в… |
| T6 | Замінник | find_substitute / find_substitute | ✅ | Можеш замінити вершкове масло у кексах на:  1. **Рослинна олія** — використай ¾ від маси масла. Це підходить для кексів, але смак буде менш вираженим.     2. **… |
| T7 | Ланцюжок інструментів | search_recipes, find_substitute / find_substitute, search_recipes | ✅ | Для приготування сирників без яєць, можна використати один з таких замінників:  1. **1 ст. л. меленого льону + 3 ст. л. води** (настояти 10 хв) — для щільнішої … |
| T8 | Рецепта немає в базі | search_recipes / search_recipes | ✅ | На жаль, у базі немає рецепту тайського супу том ям. Можу запропонувати приготувати інший суп або уточнити, які інгредієнти у вас є, щоб знайти альтернативу.… |
| T9 | Поза межами (тема) | — / — | ✅ | На жаль, я не можу допомогти з прогнозом погоди. Але можу підказати, що приготувати на сніданок або обід! Чим можу допомогти на кухні?… |
| T10 | Поза межами (медицина) | — / — | ✅ | На жаль, я не можу допомогти з лікувальними дієтами. Рекомендую звернутися до лікаря або дієтолога. Якщо вам потрібен рецепт або порада щодо приготування страв,… |
| T11 | Неоднозначний запит | — / — | ✅ | Щоб допомогти вам, мені потрібно знати, яку саме одиницю ви хочете конвертувати в грами. Наприклад, це може бути склянка, ложка або інша міра. Напишіть, будь ла… |
| T13 | Склянка ≠ cup | convert_units / convert_units | ✅ | У 2 склянках борошна приблизно 265 грамів.… |
| T12 | Контекст діалогу | search_recipes, find_substitute / find_substitute, search_recipes, convert_units | ✅ | Для половини порції вам потрібно 100 г борошна. Якщо виникнуть ще запитання, звертайтеся!… |

## 9. Підсумкова продуктова таблиця

| Поле | Відповідь |
|---|---|
| **Продукт** | ChefBot — кулінарний асистент для домашньої кухні |
| **Користувач** | Людина, яка готує вдома для себе й родини: має обмежений набір продуктів, рецепти з різних джерел (зокрема англомовних) і іноді — харчові обмеження в сім'ї |
| **Ключова задача** | Швидко відповісти на практичне питання під час готування: що приготувати з наявного, як перевести міри, чим замінити відсутній продукт |
| **Роль моделі** | Інтерпретує запит, обирає інструмент і параметри, тримає контекст діалогу (страва, порції, алергії), формулює відповідь з даних інструментів |
| **Роль системи (інструменти)** | `search_recipes` — перевірені рецепти; `convert_units` — точні розрахунки мір і температури з урахуванням густини; `find_substitute` — замінники з пропорцією |
| **Контекст / пам'ять** | Список `messages` з історією реплік і викликів інструментів; обмеження довжини 30 повідомлень |
| **Межі продукту** | Лише кулінарія; без медичних і дієтологічних порад; без вигаданих рецептів і пропорцій; без небезпечних технік |
| **Чому тут потрібен AI** | Звичайний пошук не зрозуміє «у мене алергія на яйця, а скільки грамів на половину порції?» — це поєднання трьох задач і контексту попередніх реплік. AI склеює їх в один сценарій; точність при цьому забезпечують інструменти |
| **Що показало тестування** | 12 сценаріїв: 7 ✅, 5 ⚠️, 0 ❌. Інструменти обрано правильно в 11 з 12 випадків, усі розрахунки через інструменти точні (254.4 г, 177 °C, 100 г). Відхилення: модель додає власні цифри й страви, коли інструмент нічого не знайшов (T5, T8); не завжди поєднує пошук рецепта із заміною (T7); при масштабуванні порцій без рецепта в контексті порахувала від вигаданої бази; у медичній темі не скерувала до фахівця (T10) |
| **Слабке місце** | Обмежена база рецептів і замінників: на запит поза базою бот чесно відмовляє, але для користувача це виглядає як «не знає простих речей». Друге — алергії живуть лише в історії діалогу і губляться в новій сесії або після обрізання контексту |
| **Наступний крок** | *Оновлення: профіль родини з фільтром алергенів, масштабування порцій, КБЖВ, соуси, напої, покупки й валідатор чисел реалізовано в розділі 10.* Початковий план:  Посилити правила промпту щодо «не доповнювати від себе» та ланцюжка «рецепт → заміна»; додати інструмент `scale_recipe` для порцій; підключити зовнішню базу рецептів (API або RAG по кулінарних джерелах); винести алергії й уподобання в профіль користувача (довготривала пам'ять); додати інструмент масштабування рецепта на N порцій |

## Висновки

1. **Межа між моделлю і системою спрацювала як задумано.** Модель не рахує і не вигадує рецептів — вона диспетчер і редактор. Усе, що має бути точним (грами, градуси, пропорції замін), робить код, і це перевіряється тестами T3–T5.
2. **Найважливіше правило промпту — що робити, коли даних немає.** Сценарії T5 і T8 показують, чи бот здатен сказати «не знаю» замість правдоподібної вигадки. Для кулінарного продукту це питання довіри: одна вигадана пропорція — і користувач більше не повірить жодній.
3. **Контекст через `messages` закриває типовий кухонний діалог** («а без яєць?», «а на дві порції?»), але не підходить для фактів, які мають жити довше за сесію, — алергії треба зберігати окремо.
4. **Тестування LLM-системи — це не лише «чи правильна відповідь»,** а окремо: чи викликано правильний інструмент, чи правильно використано його результат, чи дотримано меж.

## 10. ChefBot+ — розширення: родина, точні порції, КБЖВ, соуси, напої, покупки, перевірка чисел

Після тестування (розділи 7–9) лишилися слабкі місця: алергії жили лише в історії діалогу, порції модель рахувала сама, а коли даних не було, модель домальовувала цифри. Тут ці рішення перенесено **в код**. Це та сама логіка, що працює у веб-ChefBot і Streamlit-версії.

| Що додано | Де живе | Чому не модель |
|---|---|---|
| Профіль родини й фільтр алергенів | `FamilyProfile` + `recipe_allergens()` | Безпека не може залежати від уважності моделі; профіль не губиться при обрізанні історії |
| `get_recipe` з масштабуванням порцій | код | Порції — це арифметика; раніше модель рахувала від вигаданої бази |
| `get_nutrition` (ккал і БЖВ на порцію) | таблиця продуктів на 100 г | Цифри мають бути відтворюваними, з указанням джерела |
| `get_sauce`, `pair_drinks` | бібліотеки соусів і напоїв | Підбір із перевіреної бази, завжди з безалкогольним варіантом |
| `add_to_shopping` | список покупок у системі | Дія, а не текст: список можна показати, злити, очистити |
| `verify_answer` — валідатор чисел | код після відповіді моделі | Число з одиницею, якого немає в даних інструментів, позначається ⚠ |

База розширена до **33 рецептів**, 8 соусів і 14 напоїв (ті самі дані, що в застосунку).

In [46]:
%%writefile chefbot_plus_data.json
{"recipes":[{"name":"Запечена курка з картоплею","time":"50 хв","servings":4,"category":"Щодня","description":"Соковиті стегна з картоплею та часником у духовці.","ingredients":["курка","картопля","часник","олія","цибуля"],"amounts":null,"vegetarian":false,"gluten_free":true,"steps":["Наріжте картоплю часточками, цибулю — кільцями.","Змішайте з олією, часником, сіллю та перцем.","Викладіть курку зверху, запікайте 45 хв при 200 °C."],"nuts":false},{"name":"Курячий суп з рисом","time":"40 хв","servings":4,"category":"Щодня","description":"Легкий домашній суп, що зігріває.","ingredients":["курка","рис","морква","цибуля","картопля"],"amounts":null,"vegetarian":false,"gluten_free":true,"steps":["Зваріть бульйон з курки 20 хв.","Додайте картоплю, моркву й цибулю.","Всипте рис і варіть ще 15 хв."],"nuts":false},{"name":"Рис з овочами по-домашньому","time":"25 хв","servings":4,"category":"Щодня","description":"Швидкий вегетаріанський гарнір або окрема страва.","ingredients":["рис","морква","цибуля","часник","помідори","олія"],"amounts":null,"vegetarian":true,"gluten_free":true,"steps":["Обсмажте цибулю й моркву на олії.","Додайте помідори й часник, тушкуйте 5 хв.","Змішайте з відвареним рисом."],"nuts":false},{"name":"Омлет із сиром і помідорами","time":"10 хв","servings":4,"category":"Щодня","description":"Ситний сніданок за кілька хвилин.","ingredients":["яйця","молоко","сир","помідори","олія"],"amounts":null,"vegetarian":true,"gluten_free":true,"steps":["Збийте яйця з молоком, посоліть.","Вилийте на розігріту сковорідку з олією.","Додайте помідори та тертий сир, накрийте кришкою на 3 хв."],"nuts":false},{"name":"Макарони з томатним соусом","time":"20 хв","servings":4,"category":"Щодня","description":"Класика на кожен день.","ingredients":["макарони","помідори","часник","цибуля","олія","сир"],"amounts":null,"vegetarian":true,"gluten_free":false,"steps":["Відваріть макарони до стану al dente.","Обсмажте цибулю й часник, додайте помідори, тушкуйте 10 хв.","Змішайте з макаронами, посипте сиром."],"nuts":false},{"name":"Деруни","time":"30 хв","servings":4,"category":"Щодня","description":"Хрусткі картопляні пляцки, найкращі зі сметаною.","ingredients":["картопля","цибуля","яйця","борошно","олія"],"amounts":null,"vegetarian":true,"gluten_free":false,"steps":["Натріть картоплю й цибулю, відтисніть зайву рідину.","Додайте яйце, борошно, сіль.","Смажте на олії до золотистої скоринки з обох боків."],"nuts":false},{"name":"Тушкована яловичина з овочами","time":"1 год 30 хв","servings":4,"category":"Щодня","description":"Ніжне м'ясо в густій овочевій підливі.","ingredients":["яловичина","картопля","морква","цибуля","помідори","часник"],"amounts":null,"vegetarian":false,"gluten_free":true,"steps":["Обсмажте шматочки яловичини до скоринки.","Додайте цибулю, моркву, помідори та трохи води.","Тушкуйте 1 год, додайте картоплю ще на 25 хв."],"nuts":false},{"name":"Риба, запечена з овочами","time":"35 хв","servings":4,"category":"Щодня","description":"Легка страва з білою рибою та овочами.","ingredients":["риба","морква","цибуля","помідори","олія","часник"],"amounts":null,"vegetarian":false,"gluten_free":true,"steps":["Викладіть овочі в форму, полийте олією.","Покладіть рибу зверху, приправте часником і сіллю.","Запікайте 25 хв при 190 °C."],"nuts":false},{"name":"Паста з креветками у вершковому соусі","time":"25 хв","servings":1,"category":"Вечеря","description":"Фетучині з креветками, шпинатом і пармезаном — ресторанна вечеря вдома.","ingredients":["макарони","креветки","вершки","шпинат","сир","часник","лимон","вершкове масло","олія"],"amounts":[["Фетучині","120 г"],["Креветки","180 г"],["Вершки 20%","150 г"],["Шпинат","40 г"],["Пармезан","35 г"],["Часник","8 г"],["Лимон","30 г"],["Вершкове масло","20 г"],["Оливкова олія","10 г"],["Зелень","щіпка"]],"vegetarian":false,"gluten_free":false,"steps":["Відваріть фетучині до стану al dente.","Обсмажте креветки з часником на оливковій олії й вершковому маслі 2–3 хв.","Влийте вершки, додайте шпинат і сік лимона, прогрійте 2 хв.","Змішайте з пастою, посипте пармезаном і зеленню. Вихід — близько 420 г."],"nuts":false},{"name":"Креветки у вершковому соусі","time":"15 хв","servings":3,"category":"Вечеря","description":"Креветки в часниково-вершковому соусі з пармезаном, подавати з хлібом.","ingredients":["креветки","вершки","часник","сир","вершкове масло","лимон"],"amounts":[["Креветки","500 г"],["Вершки 20–30%","200 мл"],["Часник","3 зубчики"],["Пармезан тертий","50 г"],["Вершкове масло","2 ст. л."],["Сік лимона","½ лимона"],["Сіль, перець, зелень","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Обсмажте креветки на маслі 2–3 хв.","Додайте часник, влийте вершки й лимонний сік.","Тушкуйте 5–7 хв, додайте пармезан, сіль і перець.","Посипте зеленню, подавайте з хлібом."],"nuts":false},{"name":"Лосось у медово-гірчичному соусі","time":"25 хв","servings":3,"category":"Вечеря","description":"Глазурований лосось зі спаржею — швидко, корисно й святково.","ingredients":["лосось","мед","гірчиця","соєвий соус","олія","часник","лимон"],"amounts":[["Філе лосося","500 г"],["Мед","1 ст. л."],["Діжонська гірчиця","1 ст. л."],["Соєвий соус","1 ст. л."],["Оливкова олія","1 ст. л."],["Часник","1 зубчик"],["Сік лимона","1 ч. л."],["Сіль, перець, зелень","до смаку"]],"vegetarian":false,"gluten_free":false,"steps":["Змішайте мед, гірчицю, соєвий соус, олію, часник і лимонний сік.","Залийте лосось двома третинами соусу на 15 хв.","Обсмажте на середньому вогні по 3–4 хв з кожного боку.","Влийте решту соусу, прогрійте 1–2 хв; подавайте зі спаржею або рисом."],"nuts":false},{"name":"Лосось у вершковому маринаді","time":"45 хв","servings":2,"category":"Вечеря","description":"Запечений лосось у вершках з часником, гірчицею і кропом.","ingredients":["лосось","вершки","часник","лимон","вершкове масло","гірчиця","кріп"],"amounts":[["Лосось","2 шматки"],["Вершки","100 мл"],["Часник","2 зубчики"],["Сік лимона","1 ст. л."],["Вершкове масло","20 г"],["Гірчиця","1 ч. л."],["Кріп, сіль, перець","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Змішайте вершки, часник, лимонний сік, гірчицю і кріп.","Замаринуйте лосось на 20 хв.","Запікайте 25 хв при 180 °C."],"nuts":false},{"name":"Форель зі спаржею в вершково-ікорному соусі","time":"30 хв","servings":2,"category":"Святкове","description":"Авторська святкова страва: форель, спаржа і соус з червоною ікрою.","ingredients":["форель","спаржа","вершки","сир","ікра","часник","вершкове масло","олія","лимон","кріп"],"amounts":[["Філе форелі","350–400 г"],["Спаржа","180 г"],["Вершки 20%","150 мл"],["Вершковий сир","40 г"],["Червона ікра","40 г"],["Часник","1 зубчик"],["Вершкове масло","15 г"],["Оливкова олія","10 г"],["Лимонний сік","10 г"],["Кріп","5 г"]],"vegetarian":false,"gluten_free":true,"steps":["Посоліть і поперчіть форель, збризніть лимонним соком.","Обріжте спаржу на 2–3 см і обсмажте 3–4 хв.","Обсмажте форель на суміші олій по 3–4 хв з кожного боку.","Прогрійте вершки з сиром і часником до гладкості.","Зніміть з вогню, вмішайте ікру й полийте рибу."],"nuts":false},{"name":"Червона риба під сирною шапкою","time":"25 хв","servings":3,"category":"Вечеря","description":"Філе червоної риби зі шпинатом і вершковим сиром під золотистим сиром.","ingredients":["лосось","шпинат","сир","лимон"],"amounts":[["Філе червоної риби","400 г"],["Свіжий шпинат","100 г"],["Вершковий сир","100 г"],["Лимонний сік","1 ст. л."],["Твердий сир","50 г"]],"vegetarian":false,"gluten_free":true,"steps":["Збризніть філе лимонним соком.","Припустіть шпинат на сковороді.","Змішайте шпинат з вершковим сиром і викладіть на рибу.","Посипте твердим сиром.","Запікайте при 190 °C. Порція ≈ 320 ккал."],"nuts":false,"nutri":{"kcal":320,"p":30,"f":22,"c":2}},{"name":"Курка з часниково-вершковим соусом","time":"25 хв","servings":2,"category":"Вечеря","description":"Соковите куряче філе під ніжним соусом з пармезаном.","ingredients":["курка","вершки","часник","сир","вершкове масло"],"amounts":[["Куряче філе","2 шт"],["Вершки 20%","150 мл"],["Часник","3 зубчики"],["Пармезан","30 г"],["Вершкове масло","15 г"],["Чорний перець, сіль","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Обсмажте філе до золотистої скоринки.","Дрібно наріжте часник і прогрійте на 15 г масла 30 с.","Влийте 150 мл вершків, додайте пармезан.","Прогрійте соус 3–4 хв і полийте курку."],"nuts":false},{"name":"Курячий рулет зі шпинатом і сиром","time":"50 хв","servings":4,"category":"Святкове","description":"Запечений рулет з ніжною сирно-шпинатною начинкою.","ingredients":["курка","сир","шпинат","часник"],"amounts":[["Куряче філе","700 г"],["Сир","100 г"],["Шпинат","50 г"],["Часник","1 зубчик"],["Сіль","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Розріжте філе і відбийте тонким шаром.","Викладіть шпинат, сир і часник, згорніть рулетом.","Закріпіть краї, додайте спеції.","Запікайте 35–40 хв при 180 °C, дайте постояти 10 хв перед нарізанням."],"nuts":false},{"name":"Свинина в медово-гірчичному маринаді","time":"2 год 50 хв","servings":5,"category":"Святкове","description":"Запечена свинина з медовою глазур’ю для родинного столу.","ingredients":["свинина","мед","гірчиця","соєвий соус","часник","олія"],"amounts":[["Свинина","700 г"],["Мед","2 ст. л."],["Гірчиця","2 ч. л."],["Соєвий соус","2 ст. л."],["Часник","3 зубчики"],["Оливкова олія","1–2 ст. л."],["Паприка","1 ч. л."],["Перець, сіль","до смаку"]],"vegetarian":false,"gluten_free":false,"steps":["Змішайте мед, гірчицю, соєвий соус, часник, олію і паприку.","Маринуйте свинину 1–2 год.","Запікайте 50 хв при 180 °C."],"nuts":false},{"name":"Стейк з перцевим соусом","time":"20 хв","servings":2,"category":"Вечеря","description":"Класичний соус з чорним перцем горошком і гірчицею до стейка.","ingredients":["яловичина","вершки","вершкове масло","гірчиця","бульйон"],"amounts":[["Стейк","2 шт"],["Вершки 20%","150 мл"],["Чорний перець горошком","1 ч. л."],["Вершкове масло","15 г"],["Гірчиця","1 ч. л."],["М’ясний бульйон","50 мл"],["Сіль","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Злегка подавіть перець і прогрійте на вершковому маслі.","Влийте бульйон і доведіть до кипіння.","Додайте вершки, потім гірчицю.","Уваріть 4–5 хв до загустіння й полийте стейк."],"nuts":false},{"name":"Тефтелі у вершково-томатному соусі","time":"45 хв","servings":4,"category":"Родинне","description":"Ніжні тефтелі з рисом у густому соусі — улюблені дітьми.","ingredients":["фарш","рис","цибуля","яйця","вершки","томатна паста","борошно","олія"],"amounts":[["Фарш змішаний","600 г"],["Рис відварний","120 г"],["Цибуля","100 г"],["Яйце","1 шт"],["Вершки 20%","180 мл"],["Томатна паста","60 г"],["Вода","250 мл"],["Борошно","1 ст. л."],["Олія","30 мл"],["Петрушка","15 г"]],"vegetarian":false,"gluten_free":false,"steps":["Відваріть рис до напівготовності й остудіть.","Змішайте фарш, рис, цибулю, яйце, сіль і перець.","Сформуйте тефтелі діаметром 4 см і обсмажте до легкої скоринки.","Змішайте томатну пасту, вершки, воду й борошно.","Залийте тефтелі соусом і тушкуйте під кришкою 20–25 хв."],"nuts":false},{"name":"Голубці","time":"1 год 30 хв","servings":6,"category":"Родинне","description":"Ніжні голубці з м’ясом і рисом у томатно-сметанному соусі.","ingredients":["капуста","фарш","рис","цибуля","морква","томатна паста","сметана","олія"],"amounts":[["Капуста","1 качан"],["Фарш свино-яловичий","500 г"],["Рис","1 склянка"],["Цибуля","1 шт"],["Морква","1 шт"],["Томатна паста","2 ст. л."],["Вода або бульйон","500 мл"],["Сметана","2 ст. л."],["Лавровий лист, сіль, перець","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Відваріть капусту в підсоленій воді й обережно зніміть листя.","Змішайте фарш з рисом і цибулею, посоліть.","Загорніть начинку в листя конвертиком.","Викладіть на моркву з цибулею.","Залийте соусом з томатної пасти, сметани й води.","Тушкуйте під кришкою 40–60 хв."],"nuts":false},{"name":"Фаршировані перці з м’ясом і рисом","time":"1 год 15 хв","servings":6,"category":"Родинне","description":"Перці з соковитою начинкою, тушковані в томатному соусі.","ingredients":["перець","фарш","рис","цибуля","морква","томати","сметана","часник","олія"],"amounts":[["Солодкий перець","6 шт"],["Фарш яловичий або змішаний","600 г"],["Рис відварний","180 г"],["Цибуля","120 г"],["Морква","120 г"],["Томатна пасата","350 мл"],["Сметана","100 г"],["Часник","2 зубчики"],["Олія","30 мл"],["Петрушка","15 г"]],"vegetarian":false,"gluten_free":true,"steps":["Підготуйте перці, відваріть рис до напівготовності.","Змішайте фарш, рис, цибулю й моркву.","Наповніть перці, не утрамбовуючи щільно.","Зробіть соус з пасати, сметани й часнику.","Тушкуйте на слабкому вогні."],"nuts":false},{"name":"Стейк з цвітної капусти з пюре з квасолі","time":"45 хв","servings":3,"category":"Овочеве","description":"Запечені стейки з капусти на кремовому пюре з білої квасолі з чимічурі.","ingredients":["цвітна капуста","квасоля","олія","часник","лимон","паприка"],"amounts":[["Цвітна капуста","1 велика головка"],["Оливкова олія","3 ст. л."],["Паприка","1 ч. л."],["Часниковий порошок","½ ч. л."],["Біла квасоля","1 банка (400 г)"],["Часник","1 зубчик"],["Лимонний сік","2 ст. л."],["Соус чимічурі","до подачі"]],"vegetarian":true,"gluten_free":true,"steps":["Наріжте капусту на 2–3 стейки товщиною 2,5 см.","Змастіть олією, посипте паприкою, часником, сіллю й перцем.","Обсмажте по 4–5 хв з кожного боку.","Запікайте 10 хв при 200 °C.","Збийте квасолю з часником, олією і лимонним соком у пюре, подавайте зі стейком і чимічурі."],"nuts":false},{"name":"Хрусткі картопляні дольки по-сільськи","time":"1 год","servings":4,"category":"Гарнір","description":"Запечені дольки з паприкою, часником і розмарином.","ingredients":["картопля","олія","паприка","часник","розмарин"],"amounts":[["Картопля","1 кг"],["Рослинна олія","45 мл"],["Солодка паприка","2 ч. л."],["Сушений часник","1 ч. л."],["Сушений розмарин","1 ч. л."],["Сіль","1¼ ч. л."],["Чорний перець","½ ч. л."]],"vegetarian":true,"gluten_free":true,"steps":["Помийте картоплю й розріжте на 8 дольок.","Замочіть на 20 хв і дуже добре обсушіть.","Змішайте олію зі спеціями й покрийте картоплю.","Викладіть в один шар і запікайте 35–40 хв, перевернувши через 20 хв."],"nuts":false},{"name":"Картопляні рулетики з куркою","time":"40 хв","servings":4,"category":"Родинне","description":"Рум’яні рулетики з пюре з ніжною куряче-сирною начинкою.","ingredients":["картопля","курка","сир","яйця","кріп"],"amounts":[["Картопляне пюре","500 г"],["Курка кубиками","180 г"],["Сир дрібно","90 г"],["Яйце","1 шт"],["Кріп","1 ст. л."]],"vegetarian":false,"gluten_free":true,"steps":["Змішайте пюре з яйцем.","Курку з’єднайте з сиром і кропом.","Викладіть начинку на пюре й згорніть рулетики.","Запечіть до рум’яної скоринки."],"nuts":false},{"name":"Курячі сердечка з грибами у вершках","time":"40 хв","servings":4,"category":"Родинне","description":"Ніжні сердечка з печерицями в вершковому соусі.","ingredients":["курячі сердечка","гриби","вершки","цибуля","часник","олія"],"amounts":[["Курячі сердечка","600 г"],["Печериці","300 г"],["Вершки 20%","200 мл"],["Цибуля","1 шт"],["Часник","2 зубчики"],["Олія","1 ст. л."],["Сіль, перець","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["Обсмажте сердечка на олії 7–8 хв.","Додайте гриби й цибулю, смажте ще 5–7 хв.","Влийте вершки, додайте часник, сіль і перець.","Тушкуйте під кришкою 20 хв на слабкому вогні."],"nuts":false},{"name":"Салат з манго, авокадо і креветками","time":"10 хв","servings":1,"category":"Легке","description":"Тропічний ресторанний салат. ≈190 ккал на порцію.","ingredients":["манго","авокадо","креветки","салат","лайм"],"amounts":[["Манго","100 г"],["Авокадо","70 г"],["Креветки","100 г"],["Мікс салатів","40 г"],["Сік лайма","до смаку"]],"vegetarian":false,"gluten_free":true,"steps":["З’єднайте всі інгредієнти.","Збризніть соком лайма.","Обережно перемішайте."],"nuts":false,"nutri":{"kcal":190,"p":13,"f":8,"c":16}},{"name":"Салат з грушею, горгонзолою і волоськими горіхами","time":"10 хв","servings":1,"category":"Легке","description":"Солодка груша, пікантний сир і хрусткі горіхи. ≈180 ккал. Містить горіхи!","ingredients":["груша","сир","горіхи","рукола"],"amounts":[["Груша","150 г"],["Горгонзола","30 г"],["Волоські горіхи","15 г"],["Рукола","40 г"]],"vegetarian":true,"gluten_free":true,"steps":["Змішайте руколу й скибочки груші.","Додайте сир і горіхи.","Подавайте одразу."],"nuts":true,"nutri":{"kcal":180,"p":6,"f":11,"c":15}},{"name":"Салат «Опівночі в Парижі»","time":"20 хв","servings":3,"category":"Святкове","description":"Ростбіф, печериці, рукола і в’ялені томати з гірчично-бальзамічною заправкою.","ingredients":["яловичина","гриби","рукола","томати","сир","гірчиця"],"amounts":[["Яловичий ростбіф","300 г"],["Печериці","250 г"],["Рукола","100 г"],["В’ялені томати","100 г"],["Пармезан","50 г"],["Гірчично-бальзамічна заправка","3 ст. л."]],"vegetarian":false,"gluten_free":true,"steps":["Обсмажте печериці до рум’яності.","Змішайте з руколою і томатами.","Додайте скибочки ростбіфа й пармезан, полийте заправкою."],"nuts":false},{"name":"Сирники з персиками і ванільним кремом","time":"30 хв","servings":4,"category":"Сніданок","description":"Сирники з соковитою начинкою з персиків і ванільного крему.","ingredients":["сир кисломолочний","яйця","цукор","борошно","персики","крем"],"amounts":[["Кисломолочний сир 5%","500 г"],["Яйце","1 шт"],["Цукор","2 ст. л."],["Борошно","3 ст. л."],["Персики","2 шт"],["Ванільний пудинг або заварний крем","150 г"],["Ванільний цукор","1 ч. л."]],"vegetarian":true,"gluten_free":false,"steps":["Приготуйте сирну масу.","Наріжте персики невеликими кубиками.","У центр покладіть ванільний крем і персики.","Сформуйте сирники й обсмажте до рум’яної скоринки.","Подавайте теплими з цукровою пудрою."],"nuts":false},{"name":"Японські суфле-панкейки","time":"30 хв","servings":2,"category":"Сніданок","description":"Високі й повітряні панкейки — для неспішного недільного ранку.","ingredients":["яйця","молоко","борошно","крохмаль","цукор","олія"],"amounts":[["Яєчний жовток","20 г (1 шт)"],["Молоко","20 г"],["Борошно пшеничне","40 г"],["Кукурудзяний крохмаль","10 г"],["Розпушувач","2 г (½ ч. л.)"],["Яєчний білок","60 г (2 шт)"],["Цукор","30 г"],["Ванільний екстракт","до смаку"]],"vegetarian":true,"gluten_free":false,"steps":["Збийте жовток з молоком і ваніллю.","Просійте борошно, крохмаль і розпушувач, перемішайте.","Збийте білки з цукром до стійких піків.","Обережно з’єднайте в 2–3 прийоми.","Відсадіть на змащену сковороду під кришкою.","Готуйте на мінімальному вогні 5–6 хв, переверніть і ще 4–5 хв."],"nuts":false},{"name":"Суфле з сиру і банана","time":"5 хв","servings":1,"category":"Сніданок","description":"Легкий сніданок у мікрохвильовці за кілька хвилин.","ingredients":["сир кисломолочний","банан","яйця"],"amounts":[["Кисломолочний сир","180 г"],["Банан","80 г"],["Яйце","1 шт"]],"vegetarian":true,"gluten_free":true,"steps":["Збийте сир, банан і яйце.","Готуйте в мікрохвильовці 4 хв на високій потужності.","Дайте трохи остигнути."],"nuts":false},{"name":"Кабачкові мафіни","time":"35 хв","servings":6,"category":"Сніданок","description":"Ситні мафіни з кабачком, сиром і зеленню.","ingredients":["кабачок","яйця","борошно","сир","розпушувач","зелень"],"amounts":[["Кабачок","250 г"],["Яйця","2 шт"],["Борошно","150 г"],["Сир","120 г"],["Розпушувач","1 ч. л."],["Зелень","20 г"]],"vegetarian":true,"gluten_free":false,"steps":["Натріть кабачок і відтисніть сік.","Змішайте з яйцями, сиром, зеленню, борошном і розпушувачем.","Розкладіть у форми й випікайте приблизно 25 хв при 180 °C."],"nuts":false},{"name":"Синабони з корицею","time":"3 год 30 хв","servings":12,"category":"Десерт","description":"М’які булочки з корицею під ніжним сирним кремом.","ingredients":["молоко","дріжджі","цукор","яйця","вершкове масло","борошно","кориця","вершковий сир","цукрова пудра"],"amounts":[["Молоко","250 мл"],["Сухі дріжджі","11 г"],["Цукор (тісто)","130 г + 2 ч. л."],["Яйця","2 шт"],["Вершкове масло (тісто)","80 г"],["Борошно","близько 600 г"],["Масло (начинка)","80 г"],["Цукор (начинка)","200 г"],["Кориця","20 г"],["Вершковий сир (крем)","150 г"],["Масло (крем)","80 г"],["Цукрова пудра","100 г"]],"vegetarian":true,"gluten_free":false,"steps":["Змішайте тепле молоко, дріжджі й 2 ч. л. цукру, залиште на 20 хв.","Змішайте яйця, 130 г цукру, сіль, опару і 80 г розтопленого масла.","Додайте борошно, замісіть тісто й залиште на 1,5 год.","Розкачайте, змастіть маслом і посипте цукром з корицею.","Зверніть рулет, наріжте на 12 частин і дайте постояти 30–40 хв.","Випікайте 20–25 хв при 180 °C.","Змішайте сир, масло й пудру і нанесіть на теплі булочки."],"nuts":false}],"sauces":[{"name":"Часниково-вершковий соус","for":"курка, паста, гриби","time":"6 хв","amounts":[["Вершки 20%","150 мл"],["Часник","3 зубчики"],["Пармезан","30 г"],["Вершкове масло","15 г"],["Чорний перець, сіль","до смаку"]],"steps":["Дрібно наріжте часник.","Розтопіть масло й прогрійте часник 30 с.","Влийте вершки, додайте пармезан.","Прогрійте 3–4 хв до загустіння."]},{"name":"Перцевий соус до стейка","for":"яловичина, свинина","time":"10 хв","amounts":[["Вершки 20%","150 мл"],["Чорний перець горошком","1 ч. л."],["Вершкове масло","15 г"],["Гірчиця","1 ч. л."],["М’ясний бульйон","50 мл"],["Сіль","до смаку"]],"steps":["Злегка подавіть перець.","Прогрійте його у вершковому маслі.","Влийте бульйон і доведіть до кипіння.","Додайте вершки, потім гірчицю, уваріть 4–5 хв."]},{"name":"Чимічурі","for":"стейк, цвітна капуста, курка на грилі","time":"10 хв","amounts":[["Петрушка","1 пучок (≈ 40 г)"],["Часник","2 зубчики"],["Сушений орегано","1 ч. л."],["Червоний винний оцет","2 ст. л."],["Оливкова олія","100 мл"],["Пластівці чилі","щіпка"],["Сіль","½ ч. л."]],"steps":["Дрібно посічіть петрушку й часник.","Змішайте з орегано, чилі та сіллю.","Влийте оцет і олію, перемішайте.","Дайте настоятися 15 хв."]},{"name":"Медово-гірчичний соус","for":"лосось, свинина, курка","time":"3 хв","amounts":[["Мед","1 ст. л."],["Діжонська гірчиця","1 ст. л."],["Соєвий соус","1 ст. л."],["Оливкова олія","1 ст. л."],["Часник","1 зубчик"],["Сік лимона","1 ч. л."]],"steps":["Змішайте всі інгредієнти до однорідності.","Дві третини — для маринаду, решту — полити страву наприкінці."]},{"name":"Вершково-ікорний соус","for":"форель, лосось, спаржа","time":"5 хв","amounts":[["Вершки 20%","150 мл"],["Вершковий сир","40 г"],["Червона ікра","40 г"],["Часник","1 зубчик"],["Кріп","5 г"]],"steps":["Прогрійте вершки з сиром і часником до гладкості.","Зніміть з вогню.","Вмішайте ікру й кріп — ікру не нагрівайте."]},{"name":"Вершково-томатний соус","for":"тефтелі, макарони, курка","time":"8 хв","amounts":[["Томатна паста","60 г"],["Вершки 20%","180 мл"],["Вода","250 мл"],["Борошно","1 ст. л."],["Сіль, перець","до смаку"]],"steps":["Розмішайте борошно у воді без грудочок.","Додайте томатну пасту й вершки.","Прогрійте на слабкому вогні до загустіння."]},{"name":"Сирний крем для випічки","for":"синабони, мафіни, торти","time":"5 хв","amounts":[["Вершковий сир","150 г"],["М’яке вершкове масло","80 г"],["Цукрова пудра","100 г"]],"steps":["Збийте м’яке масло до пухкості.","Додайте сир і пудру, збийте до гладкості.","Наносьте на теплу, а не гарячу випічку."]},{"name":"Гірчично-бальзамічна заправка","for":"салати з ростбіфом, руколою, грушею","time":"2 хв","amounts":[["Оливкова олія","3 ст. л."],["Бальзамічний оцет","1 ст. л."],["Діжонська гірчиця","1 ч. л."],["Мед","½ ч. л."],["Сіль, перець","до смаку"]],"steps":["Змішайте гірчицю, мед і оцет.","Влийте олію тонкою цівкою, збиваючи виделкою."]}],"drinks":[{"group":"wine","name":"Біле сухе (совіньйон блан)","temp":"8–10 °C","note":"Свіже, з цитрусовою кислинкою — підкреслює морепродукти й зелень.","pairs":["креветки","форель","риба","спаржа","салат","лайм","манго"]},{"group":"wine","name":"Шардоне","temp":"10–12 °C","note":"Округле й вершкове — у парі з вершковими соусами.","pairs":["вершки","лосось","курка","гриби","сир"]},{"group":"wine","name":"Рожеве сухе","temp":"8–10 °C","note":"Легке ягідне — до літніх страв, перців і томатів.","pairs":["перець","томати","помідори","груша","рукола"]},{"group":"wine","name":"Червоне легке (піно нуар)","temp":"14–16 °C","note":"М’які танини — до лосося в глазурі, грибів і тефтелів.","pairs":["мед","гриби","фарш","курячі сердечка"]},{"group":"wine","name":"Червоне насичене (каберне)","temp":"16–18 °C","note":"Глибоке й структурне — до стейка, свинини, тушкованого м’яса.","pairs":["яловичина","свинина","бульйон"]},{"group":"wine","name":"Ігристе брют","temp":"6–8 °C","note":"Святкові бульбашки — до ікри, канапе й святкового столу.","pairs":["ікра","Святкове"]},{"group":"fresh","name":"Лимонад з огірком і м’ятою","temp":"4–6 °C","note":"Прохолодний і трав’яний — до салатів, риби й морепродуктів.","pairs":["лайм","авокадо","креветки","салат","Легке"]},{"group":"fresh","name":"Газована вода з лимоном","temp":"6–8 °C","note":"Нейтральна — очищає смак між стравами, пасує до всього.","pairs":["*"]},{"group":"fresh","name":"Журавлиний морс","temp":"8–10 °C","note":"Кисло-солодкий — освіжає жирні м’ясні страви.","pairs":["свинина","яловичина","фарш","Родинне"]},{"group":"tea","name":"Чорний чай з лимоном","temp":"80–90 °C","note":"Класика до сніданку, сирників і дерунів.","pairs":["Сніданок","сир кисломолочний","картопля","яйця"]},{"group":"tea","name":"Зелений чай","temp":"75–80 °C","note":"Делікатний — до риби, рису й легких страв.","pairs":["рис","риба","лосось","Легке"]},{"group":"tea","name":"Еспресо або капучино","temp":"—","note":"Гірчинка кави врівноважує солодке — до випічки й десертів.","pairs":["Десерт","кориця","цукрова пудра"]},{"group":"home","name":"Вишневий компот","temp":"охолоджений","note":"Родинний смак — до супів, голубців і буденних обідів.","pairs":["Родинне","Щодня","капуста"]},{"group":"home","name":"Узвар із сухофруктів","temp":"теплий або охолоджений","note":"Димно-фруктовий — до святкового столу, свинини й випічки.","pairs":["Святкове","свинина","Десерт","мед"]}]}

Overwriting chefbot_plus_data.json


In [47]:
import json, re
from fractions import Fraction

PLUS = json.load(open("chefbot_plus_data.json", encoding="utf-8"))
RECIPES_PLUS, SAUCES, DRINKS = PLUS["recipes"], PLUS["sauces"], PLUS["drinks"]
print(f"Рецептів: {len(RECIPES_PLUS)} · соусів: {len(SAUCES)} · напоїв: {len(DRINKS)}")

# ---------- Алергени: ключові слова за початком слова ----------
ALLERGENS = {
    "горіхи":       {"kw": ["горіх", "фундук", "мигдал", "кешью", "арахіс", "фісташ", "пекан"], "flag": lambda r: bool(r.get("nuts"))},
    "глютен":       {"kw": [], "flag": lambda r: not r.get("gluten_free")},
    "молоко":       {"kw": ["молок", "вершк", "сир", "сметан", "йогурт", "пармезан", "горгонзол", "крем", "пудинг"]},
    "яйця":         {"kw": ["яйц", "яйце", "жовток", "білок"]},
    "риба":         {"kw": ["лосос", "форел", "риба", "рибн", "ікр", "тунец", "оселед"]},
    "морепродукти": {"kw": ["кревет", "кальмар", "мідії", "краб"]},
    "мед":          {"kw": ["мед"]},
}

def _words(r):
    raw = list((r.get("ingredients") or [])) + [a[0] for a in (r.get("amounts") or [])]
    return [w for x in raw for w in re.split(r"[\s,·()«»\"’'.:;/-]+", str(x).lower()) if w]

def recipe_allergens(r) -> list[str]:
    w = _words(r)
    out = []
    for name, rule in ALLERGENS.items():
        flag = rule.get("flag")
        if (flag and flag(r)) or any(x.startswith(k) for k in rule["kw"] for x in w):
            out.append(name)
    return out


class FamilyProfile:
    """Довготривала пам'ять про родину: живе поза історією діалогу і не обрізається разом з нею."""

    def __init__(self):
        self.members: list[dict] = []

    def add(self, name: str, allergies=(), vegetarian: bool = False):
        bad = [a for a in allergies if a not in ALLERGENS]
        if bad:
            raise ValueError(f"Невідомі алергени: {bad}. Доступні: {list(ALLERGENS)}")
        self.members.append({"name": name, "allergies": list(allergies), "veg": vegetarian})
        return self

    def conflicts(self, r) -> list[dict]:
        a = recipe_allergens(r)
        out = []
        for m in self.members:
            what = [x for x in m["allergies"] if x in a]
            if m["veg"] and not r.get("vegetarian"):
                what.append("не вегетаріанське")
            if what:
                out.append({"хто": m["name"], "що": what})
        return out

    def is_safe(self, r) -> bool:
        return not self.conflicts(r)

    def summary(self) -> str:
        if not self.members:
            return "не вказано"
        return "; ".join(
            f"{m['name']}: {', '.join(m['allergies'] + (['вегетаріанець'] if m['veg'] else [])) or 'без обмежень'}"
            for m in self.members)


FAMILY = FamilyProfile()
FAMILY.add("Марко", ["горіхи"]).add("Оля", ["морепродукти"])
print("Профіль родини:", FAMILY.summary())
safe = [r["name"] for r in RECIPES_PLUS if FAMILY.is_safe(r)]
print(f"Безпечних для всієї родини страв: {len(safe)} з {len(RECIPES_PLUS)}")
print("Приклад відсіяної:", next((r["name"], FAMILY.conflicts(r)) for r in RECIPES_PLUS if not FAMILY.is_safe(r)))

Рецептів: 33 · соусів: 8 · напоїв: 14
Профіль родини: Марко: горіхи; Оля: морепродукти
Безпечних для всієї родини страв: 29 з 33
Приклад відсіяної: ('Паста з креветками у вершковому соусі', [{'хто': 'Оля', 'що': ['морепродукти']}])


> **Продуктовий коментар.** Алергени визначає код за складом рецепта, а не модель. Профіль родини — окремий об'єкт: він не залежить від довжини історії і щоразу передається агенту в системний промпт, а результати пошуку фільтруються ще до того, як модель їх побачить. Обмеження: фільтр бачить лише те, що записано в рецепті (наприклад, склад покупного печива невідомий), тому в продукті це **підказка, а не гарантія** — це чесно сказано користувачу.

In [48]:
# ---------- КБЖВ: ккал, білки, жири, вуглеводи на 100 г ----------
NUT = [
    ("цукрова пудра", 399, 0, 0, 100), ("кукурудзяний крохмаль", 381, 0, 0, 91), ("крохмаль", 381, 0, 0, 91),
    ("вершковий сир", 342, 6, 34, 4), ("вершкове масло", 748, .5, 82, .8), ("масло", 748, .5, 82, .8),
    ("оливкова олія", 899, 0, 100, 0), ("олія", 899, 0, 100, 0), ("кисломолочний сир", 121, 17, 5, 1.8),
    ("сир", 380, 28, 29, 1), ("пармезан", 392, 33, 28, 0), ("горгонзола", 353, 19, 31, 1),
    ("вершки", 205, 2.8, 20, 3.7), ("сметана", 206, 2.8, 20, 3.2), ("молоко", 60, 3, 3.2, 4.7),
    ("йогурт", 66, 5, 1.5, 6), ("яйце", 157, 12.7, 11.5, .7), ("яйця", 157, 12.7, 11.5, .7),
    ("жовток", 322, 16, 27, 3.6), ("білок", 52, 11, .2, .7), ("борошно", 364, 10, 1, 76),
    ("цукор", 399, 0, 0, 100), ("мед", 329, .8, 0, 81), ("фетучині", 350, 12, 1.5, 71),
    ("макарони", 350, 12, 1.5, 71), ("рис", 344, 7, 1, 79), ("курячі сердечка", 159, 16, 10, .8),
    ("куряче філе", 113, 23.6, 1.9, .4), ("курка", 190, 19, 12, 0), ("філе", 113, 23.6, 1.9, .4),
    ("яловичин", 187, 19, 12, 0), ("ростбіф", 180, 26, 8, 0), ("стейк", 250, 26, 16, 0),
    ("свинина", 242, 16, 21, 0), ("фарш", 254, 17, 20, 0), ("лосос", 208, 20, 13, 0),
    ("червоної риби", 208, 20, 13, 0), ("форел", 141, 20, 6, 0), ("риба", 90, 19, 1, 0),
    ("кревет", 95, 20, 1, 0), ("ікра", 263, 25, 17, 4), ("картопляне пюре", 90, 2, 3, 14),
    ("картопля", 77, 2, .4, 17), ("морква", 35, 1.3, .1, 7), ("цибуля", 41, 1.4, 0, 8),
    ("часник", 149, 6, .5, 33), ("томатна паста", 82, 4.8, .5, 19), ("пасата", 30, 1.5, .2, 5),
    ("в’ялені томати", 258, 14, 3, 56), ("томати", 20, 1, .2, 4), ("помідори", 20, 1, .2, 4),
    ("капуста", 27, 1.8, .1, 4.7), ("цвітна капуста", 25, 1.9, .3, 5), ("перець", 26, 1.3, .3, 5),
    ("шпинат", 22, 2.9, .3, 2), ("печериці", 27, 4.3, 1, .1), ("гриби", 27, 4.3, 1, .1),
    ("спаржа", 20, 2.2, .1, 3.9), ("кабачок", 24, .6, .3, 4.6), ("квасоля", 90, 6, .5, 15),
    ("банан", 89, 1.1, .3, 23), ("манго", 60, .8, .4, 15), ("авокадо", 160, 2, 15, 9),
    ("груша", 57, .4, .1, 15), ("персики", 39, .9, .3, 10), ("вишні", 52, .8, .2, 11),
    ("горіхи", 654, 15, 65, 14), ("рукола", 25, 2.6, .7, 3.7), ("салат", 15, 1.4, .2, 2.9),
    ("соєвий соус", 53, 6, 0, 6.6), ("гірчиця", 140, 6, 8, 6), ("лимон", 29, 1.1, .3, 9),
    ("вода", 0, 0, 0, 0), ("бульйон", 4, .5, .2, 0),
]
PIECE = {"яйц": 55, "яйце": 55, "цибул": 90, "морк": 80, "картопл": 120, "перець": 150, "персик": 150,
         "банан": 120, "зубчик": 5, "груш": 170, "лосос": 150, "філе": 180, "стейк": 220, "помідор": 120,
         "лимон": 100, "качан": 1200, "головка": 800, "пучок": 40, "банка": 400}
FRAC = {"½": .5, "¼": .25, "¾": .75}

def _first_number(q: str):
    s = re.sub(r"(\d)?([½¼¾])", lambda m: str(int(m.group(1) or 0) + FRAC[m.group(2)]), q)
    m = re.search(r"(\d+(?:[.,]\d+)?)(?:\s*[–-]\s*(\d+(?:[.,]\d+)?))?", s)
    if not m:
        return None, s
    v = float(m.group(1).replace(",", "."))
    if m.group(2):
        v = (v + float(m.group(2).replace(",", "."))) / 2
    return v, s

def grams(name: str, qty: str):
    """Переводить кількість з рецепта в грами. 0 — «до смаку», None — не вдалося оцінити."""
    q = str(qty).lower()
    if re.search(r"смак|щіпк|подачі|^—$", q):
        return 0
    v, s = _first_number(q)
    if v is None:
        return None
    n = name.lower()
    # ложки перевіряємо першими: у Python «л» у «ст. л.» інакше сприймається як літр
    if re.search(r"ст\.?\s?л", s):  return v * (10 if re.search(r"борошн|цукор|крохмал", n) else 15)
    if re.search(r"ч\.?\s?л", s):   return v * 5
    if re.search(r"кг|\bл\b|літр", s) and "мл" not in s: return v * 1000
    if re.search(r"\bг\b|г$|мл", s):                     return v
    if re.search(r"склянк|cup", s): return v * 200
    for unit, g in [("зубчик", 5), ("пучок", 40), ("банк", 400), ("качан", 1200), ("головк", 800), ("шматок", 150)]:
        if unit in s:
            return v * g
    k = next((p for p in PIECE if p in n), None)
    return v * PIECE[k] if k else None

def nutrition(r):
    k = p = f = c = 0.0
    known = total = 0
    for name, qty in (r.get("amounts") or []):
        total += 1
        g = grams(name, qty)
        if g == 0:
            known += 1
            continue
        row = next((x for x in NUT if x[0] in name.lower()), None)
        if g is None or row is None:
            continue
        known += 1
        k += row[1] * g / 100; p += row[2] * g / 100; f += row[3] * g / 100; c += row[4] * g / 100
    if not total or known / total < .6:
        return None   # даних замало — краще чесно не відповісти, ніж вигадати
    s = r.get("servings") or 4
    return {"ккал": round(k / s / 10) * 10, "білки_г": round(p / s), "жири_г": round(f / s),
            "вуглеводи_г": round(c / s), "джерело": f"оцінка за таблицею продуктів, враховано {known} з {total} інгредієнтів"}

demo = next(r for r in RECIPES_PLUS if r.get("amounts"))
print(demo["name"], "→", nutrition(demo))

Паста з креветками у вершковому соусі → {'ккал': 1300, 'білки_г': 68, 'жири_г': 70, 'вуглеводи_г': 97, 'джерело': 'оцінка за таблицею продуктів, враховано 10 з 10 інгредієнтів'}


In [49]:
# ---------- Масштабування порцій: кожне число множить код ----------
def _fmt(x: float) -> str:
    if abs(x - round(x)) < 0.05:
        return str(int(round(x)))
    for den, sym in [(2, "½"), (4, "¼"), (4, "¾")]:
        whole, frac = divmod(x, 1)
        if abs(frac - Fraction(sym == "¾" and 3 or 1, den)) < 0.05:
            return (str(int(whole)) if whole else "") + sym
    return f"{x:.1f}".replace(".", ",")

def scale_qty(qty: str, k: float) -> str:
    q = str(qty)
    if k == 1 or re.search(r"смак|щіпк|подачі|^—$", q.lower()):
        return q
    v, s = _first_number(q)
    if v is None:
        return q
    unit = re.sub(r"^[\s\d.,½¼¾–-]+", "", s).strip()
    x = v * k
    if re.search(r"\bг\b|мл|^г$", unit) and x >= 20:
        x = round(x / 5) * 5          # граматику до 5 г — без зайвої «точності»
    return f"{_fmt(x)} {unit}".strip()

def _find(name: str, pool):
    n = name.lower().strip()
    return (next((x for x in pool if x["name"].lower() == n), None)
            or next((x for x in pool if n in x["name"].lower()), None)
            or next((x for x in pool if any(w in x["name"].lower() for w in n.split() if len(w) > 3)), None))

r = _find("сирники", RECIPES_PLUS)
print(r["name"], "| база:", r["servings"], "порц.")
for (a, q) in (r.get("amounts") or [])[:4]:
    print(f"  {a}: {q} → на 6 порцій: {scale_qty(q, 6 / r['servings'])}")

Сирники з персиками і ванільним кремом | база: 4 порц.
  Кисломолочний сир 5%: 500 г → на 6 порцій: 750 г
  Яйце: 1 шт → на 6 порцій: 1½ шт
  Цукор: 2 ст. л. → на 6 порцій: 3 ст. л.
  Борошно: 3 ст. л. → на 6 порцій: 4½ ст. л.


In [50]:
from langchain_core.tools import tool

SHOPPING: dict[str, list[str]] = {}   # продукт → кількості (система, а не текст моделі)

def _dumps(x):
    return json.dumps(x, ensure_ascii=False)


@tool("search_recipes")
def search_recipes_plus(query: str = "", ingredients: str = "") -> str:
    """Шукає рецепти в базі ChefBot (33 страви) за назвою або продуктами.

    Результат уже відфільтровано за профілем родини: страви з алергенами членів родини
    не повертаються, а перелічуються в полі excluded з причиною.

    Args:
        query: назва страви або тип («сніданок», «суп», «паста»).
        ingredients: продукти, які є вдома, через кому.
    Returns:
        JSON: до 5 страв (назва, час, порції, категорія, чого бракує) і список виключених.
    """
    q = query.lower().strip()
    have = [x.strip().lower() for x in ingredients.split(",") if x.strip()]
    scored = []
    for r in RECIPES_PLUS:
        text = (r["name"] + " " + r.get("category", "") + " " + r.get("description", "")).lower()
        s = 0
        if q:
            s += 3 * sum(w[:5] in text for w in q.split() if len(w) > 2)
        if have:
            s += sum(any(h[:5] in i.lower() for i in (r.get("ingredients") or [])) for h in have)
        if s > 0 or (not q and not have):
            scored.append((s, r))
    scored.sort(key=lambda x: -x[0])
    found, excluded = [], []
    for s, r in scored:
        c = FAMILY.conflicts(r)
        if c:
            excluded.append({"name": r["name"], "conflicts": c})
            continue
        missing = [i for i in (r.get("ingredients") or []) if have and not any(h[:5] in i.lower() for h in have)]
        found.append({"name": r["name"], "time": r["time"], "servings": r["servings"],
                      "category": r.get("category"), "missing_ingredients": missing if have else None})
    return _dumps({"found": len(found[:5]), "recipes": found[:5], "excluded": excluded[:5],
                   "family": FAMILY.summary(),
                   "message": None if found else "У базі немає підхожих і безпечних для родини страв."})


@tool
def get_recipe(dish: str, servings: int = 0) -> str:
    """Повний рецепт з бази: кількості й кроки. Якщо вказано servings — код перераховує кожну кількість.

    Args:
        dish: назва страви з бази (як повернув search_recipes).
        servings: на скільки порцій потрібен рецепт; 0 — базова кількість.
    """
    r = _find(dish, RECIPES_PLUS)
    if not r:
        return _dumps({"error": "Страви немає в базі. Спершу виклич search_recipes."})
    n = servings or r["servings"]
    k = n / r["servings"]
    return _dumps({"name": r["name"], "servings": n, "base_servings": r["servings"], "time": r["time"],
                   "amounts": [[a, scale_qty(q, k)] for a, q in (r.get("amounts") or [])], "steps": r["steps"],
                   "allergens": recipe_allergens(r), "family_conflicts": FAMILY.conflicts(r)})


@tool
def get_nutrition(dish: str) -> str:
    """Калорійність і БЖВ на одну порцію страви з бази — оцінка кодом за таблицею продуктів на 100 г.

    Args:
        dish: назва страви з бази.
    """
    r = _find(dish, RECIPES_PLUS)
    if not r:
        return _dumps({"error": "Страви немає в базі."})
    n = nutrition(r)
    return _dumps({"name": r["name"], **n} if n else {"name": r["name"], "error": "Недостатньо даних для оцінки."})


@tool
def get_sauce(query: str = "") -> str:
    """Бібліотека соусів: соус за назвою або за стравою, до якої він пасує. Без аргументу — список назв.

    Args:
        query: назва соусу або продукт/страва («курка», «паста», «лосось»).
    """
    q = query.lower().strip()
    if not q:
        return _dumps([s["name"] for s in SAUCES])
    s = next((s for s in SAUCES if q in (s["name"] + " " + s["for"]).lower()), None) or \
        next((s for s in SAUCES if any(w[:5] in (s["name"] + " " + s["for"]).lower() for w in q.split() if len(w) > 3)), None)
    return _dumps(s if s else {"error": "Такого соусу в бібліотеці немає.", "available": [x["name"] for x in SAUCES]})


@tool
def pair_drinks(dish: str) -> str:
    """Підбирає до 3 напоїв до страви з бази (вино, безалкогольні, чай/кава, домашні).
    Завжди повертає хоча б один безалкогольний варіант.

    Args:
        dish: назва страви з бази.
    """
    r = _find(dish, RECIPES_PLUS)
    if not r:
        return _dumps({"error": "Страви немає в базі."})
    keys = [x.lower() for x in (r.get("ingredients") or [])] + [r.get("category", "").lower()]
    scored = []
    for d in DRINKS:
        s = 0.5 if "*" in d["pairs"] else sum(any(k in x or x in k for x in keys) for k in d["pairs"])
        if s > 0:
            scored.append((s, d))
    picks = [d for s, d in sorted(scored, key=lambda x: -x[0])[:3]]
    if not any(d["group"] != "wine" for d in picks):
        soft = next((d for s, d in sorted(scored, key=lambda x: -x[0]) if d["group"] != "wine"),
                    next(d for d in DRINKS if d["group"] != "wine"))
        picks = picks[:2] + [soft]
    return _dumps({"dish": r["name"], "drinks": picks})


@tool
def add_to_shopping(dish: str, servings: int = 0) -> str:
    """Додає інгредієнти страви з бази до списку покупок (кількості перераховано на servings).

    Args:
        dish: назва страви з бази.
        servings: кількість порцій; 0 — базова.
    """
    r = _find(dish, RECIPES_PLUS)
    if not r:
        return _dumps({"error": "Страви немає в базі."})
    n = servings or r["servings"]
    for a, q in (r.get("amounts") or []):
        if not re.search(r"смак|подачі", q.lower()):
            SHOPPING.setdefault(a, []).append(scale_qty(q, n / r["servings"]))
    return _dumps({"added": r["name"], "servings": n, "items_in_list": len(SHOPPING)})


# Старі convert_units і find_substitute (розділ 3) лишаються — вони вже протестовані.
tools_plus = [search_recipes_plus, get_recipe, get_nutrition, get_sauce, pair_drinks, add_to_shopping,
              convert_units, find_substitute]
print("Інструменти ChefBot+:", [t.name for t in tools_plus])

Інструменти ChefBot+: ['search_recipes', 'get_recipe', 'get_nutrition', 'get_sauce', 'pair_drinks', 'add_to_shopping', 'convert_units', 'find_substitute']


In [51]:
# ---------- Валідатор: число з одиницею у відповіді має бути в даних інструментів ----------
UNITS = (r"(грам\w*|г|кілограм\w*|кг|мілілітр\w*|мл|літр\w*|л|°\s?[CFС]|градус\w*|ккал|калорі\w*"
         r"|хвилин\w*|хв|секунд\w*|с|год\w*|шт|ст\.\s?л\.|ч\.\s?л\.)")

def _nums(t: str) -> set[str]:
    t = re.sub(r"(\d)?([½¼¾])", lambda m: str(int(m.group(1) or 0) + FRAC[m.group(2)]), str(t))
    return {str(float(x.replace(",", "."))) for x in re.findall(r"\d+(?:[.,]\d+)?", t)}

def verify_answer(answer: str, tool_outputs: list[str], user_text: str = "") -> dict:
    facts = _nums(" ".join(tool_outputs)) | _nums(user_text)
    bad = []
    # «170–180 грамів»: перевіряємо обидва числа діапазону
    pattern = rf"(\d+(?:[.,]\d+)?)(?:\s*[–-]\s*(\d+(?:[.,]\d+)?))?\s*{UNITS}(?![а-яіїєґa-z])"
    for m in re.finditer(pattern, answer, flags=re.I):
        nums = [x for x in (m.group(1), m.group(2)) if x]
        if any(str(float(x.replace(",", "."))) not in facts for x in nums):
            bad.append(m.group(0))
    return {"ok": not bad, "unverified": sorted(set(bad))}

# Перевірка без моделі: відтворюємо кейс із рефлексії — інструмент сказав «густина невідома», а модель дописала «≈170 г».
tool_out = ['{"error": "Густина для кіноа невідома. Виміряйте вагою."}']
print(verify_answer("В одній склянці кіноа приблизно 170 г.", tool_out, "Скільки грамів у склянці кіноа?"))
print(verify_answer("Зазвичай в одній склянці кіноа приблизно 170-180 грамів.", tool_out, "Скільки грамів у склянці кіноа?"))  # реальна відповідь з T19
print(verify_answer("2 cups борошна — це 254,4 г.", ['{"result": 254.4, "unit": "г"}'], "2 cups борошна в грамах"))

{'ok': False, 'unverified': ['170 г']}
{'ok': False, 'unverified': ['170-180 грамів']}
{'ok': True, 'unverified': []}


### 10.1. Модульні тести нових інструментів (без моделі)

In [52]:
import pandas as pd

def J(s): return json.loads(s)

unit_rows = []
def check(tid, what, cond, detail=""):
    unit_rows.append({"ID": tid, "Перевірка": what, "Результат": "✅" if cond else "❌", "Деталі": detail})

res = J(search_recipes_plus.invoke({"query": "вечеря"}))
nut_free = all("горіхи" not in recipe_allergens(_find(x["name"], RECIPES_PLUS)) for x in res["recipes"])
check("U1", "Пошук не повертає страв з алергенами родини", nut_free and res["found"] > 0,
      f"знайдено {res['found']}, виключено {len(res['excluded'])}")

base = _find("сирники", RECIPES_PLUS)
g = J(get_recipe.invoke({"dish": "сирники", "servings": base["servings"] * 2}))
first_q = base["amounts"][0][1]
check("U2", "get_recipe ×2 подвоює кількості кодом", g["servings"] == base["servings"] * 2,
      f"{base['amounts'][0][0]}: {first_q} → {g['amounts'][0][1]}")

n = J(get_nutrition.invoke({"dish": base["name"]}))
check("U3", "КБЖВ повертає ккал і джерело оцінки", "ккал" in n and "джерело" in n, str(n)[:90])

s = J(get_sauce.invoke({"query": "паста"}))
check("U4", "Соус для пасти знайдено в бібліотеці", "name" in s, s.get("name", s.get("error")))

d = J(pair_drinks.invoke({"dish": "Паста з креветками у вершковому соусі"}))
check("U5", "Напої: є безалкогольний варіант", any(x["group"] != "wine" for x in d["drinks"]),
      ", ".join(x["name"] for x in d["drinks"]))

SHOPPING.clear()
a = J(add_to_shopping.invoke({"dish": base["name"], "servings": 2}))
check("U6", "Покупки: інгредієнти додано в список", a.get("items_in_list", 0) > 0, f"позицій: {a.get('items_in_list')}")

v = verify_answer("Приблизно 170 г.", tool_out)
check("U7", "Валідатор ловить число, якого немає в даних", not v["ok"], str(v["unverified"]))

x = J(get_recipe.invoke({"dish": "том ям"}))
check("U8", "Страви поза базою — чесна помилка, а не вигадка", "error" in x, x.get("error", ""))

unit_df = pd.DataFrame(unit_rows)
display(unit_df)
print(f"Пройдено: {(unit_df['Результат'] == '✅').sum()} з {len(unit_df)}")

,ID,Перевірка,Результат,Деталі
0,U1,Пошук не повертає страв з алергенами родини,✅,"знайдено 5, виключено 2"
1,U2,get_recipe ×2 подвоює кількості кодом,✅,Кисломолочний сир 5%: 500 г → 1000 г
2,U3,КБЖВ повертає ккал і джерело оцінки,✅,"{'name': 'Сирники з персиками і ванільним кремом', 'ккал': 250, 'білки_г': 24, 'жири_г': 8"
3,U4,Соус для пасти знайдено в бібліотеці,✅,Часниково-вершковий соус
4,U5,Напої: є безалкогольний варіант,✅,"Шардоне, Біле сухе (совіньйон блан), Лимонад з огірком і м’ятою"
5,U6,Покупки: інгредієнти додано в список,✅,позицій: 7
6,U7,"Валідатор ловить число, якого немає в даних",✅,['170 г']
7,U8,"Страви поза базою — чесна помилка, а не вигадка",✅,Страви немає в базі. Спершу виклич search_recipes.


Пройдено: 8 з 8


### 10.2. Агент ChefBot+: профіль родини в промпті + валідатор після відповіді

In [53]:
from langchain.agents import create_agent

def system_prompt_plus() -> str:
    return SYSTEM_PROMPT + f"""
## Профіль родини (довготривала пам'ять, зберігається поза діалогом)
{FAMILY.summary()}
- search_recipes уже відфільтрував небезпечні страви; не пропонуй страви з поля excluded.
- Для порцій викликай get_recipe(servings=N) — не перераховуй кількості сам.
- Калорійність — лише через get_nutrition і з позначкою, що це оцінка.
- Соуси — get_sauce, напої — pair_drinks (завжди з безалкогольним варіантом), покупки — add_to_shopping.
- Не пиши жодного числа з одиницею (г, мл, °C, ккал, хв), якого немає в результатах інструментів.
"""

agent_plus = create_agent(model=llm, tools=tools_plus, system_prompt=system_prompt_plus())


class ChefBotPlusChat(ChefBotChat):
    """Той самий діалог із розділу 6 + профіль родини в промпті + перевірка чисел у відповіді."""

    def __init__(self, max_messages: int = 12):
        super().__init__(max_messages=max_messages)   # вікно як у застосунку
        self.last_check = None

    def ask(self, text: str, verbose: bool = True) -> str:
        self.messages.append(HumanMessage(content=text))
        result = agent_plus.invoke({"messages": self.messages})
        new_part = result["messages"][len(self.messages):]
        self.messages = result["messages"]
        if len(self.messages) > self.max_messages:
            cut = self.messages[-self.max_messages:]
            while cut and not isinstance(cut[0], HumanMessage):
                cut = cut[1:]
            self.messages = cut
        answer = self.messages[-1].content
        outputs = [m.content for m in new_part if isinstance(m, ToolMessage)]
        self.last_check = verify_answer(answer, outputs, text)
        self.last_tools = used_tools(new_part)
        mark = "✓ числа звірено" if self.last_check["ok"] else f"⚠ без джерела: {self.last_check['unverified']}"
        if verbose:
            print(f"👤 {text}\n🔧 Інструменти: {self.last_tools or '—'}\n🤖 {answer}\n🔎 {mark}\n" + "-" * 80)
        return answer


SHOPPING.clear()   # прибираємо позиції, додані модульними тестами
plus = ChefBotPlusChat()
plus.ask("Що приготувати на недільний сніданок для всієї родини?")
plus.ask("Давай перший варіант, але на 6 порцій.")
plus.ask("Скільки там калорій на порцію? І що до нього випити?")
plus.ask("Додай усе в список покупок.")
print("🛒 Список покупок:", {k: v for k, v in list(SHOPPING.items())[:8]})

👤 Що приготувати на недільний сніданок для всієї родини?
🔧 Інструменти: —
🤖 Які інгредієнти у вас є вдома? Це допоможе знайти відповідний рецепт.
🔎 ✓ числа звірено
--------------------------------------------------------------------------------
👤 Давай перший варіант, але на 6 порцій.
🔧 Інструменти: ['search_recipes', 'get_recipe']
🤖 Пропоную приготувати **кабачкові мафіни**. Ось рецепт на 6 порцій:

### Інгредієнти:
- Кабачок — 250 г
- Яйця — 2 шт
- Борошно — 150 г
- Сир — 120 г
- Розпушувач — 1 ч. л.
- Зелень — 20 г

### Приготування:
1. Натріть кабачок і відтисніть сік.
2. Змішайте кабачок з яйцями, сиром, зеленню, борошном і розпушувачем.
3. Розкладіть суміш у форми й випікайте приблизно 25 хв при 180 °C.

Смачного! Якщо потрібно щось ще, запитуйте.
🔎 ✓ числа звірено
--------------------------------------------------------------------------------
👤 Скільки там калорій на порцію? І що до нього випити?
🔧 Інструменти: ['get_nutrition', 'pair_drinks']
🤖 На одну порцію кабачкових мафіні

### 10.3. Тести ChefBot+ з моделлю

In [54]:
PLUS_CASES = [
    # (id, сценарій, запит, очікувані інструменти, перевірка)
    ("T14", "Алергія з профілю", "Порадь десерт з горіхами для всієї родини", ["search_recipes"],
     "не пропонує страв з горіхами — Марко в профілі"),
    ("T15", "Порції кодом", "Дай рецепт сирників на 8 порцій", ["get_recipe"],
     "кількості з get_recipe, а не з голови"),
    ("T16", "КБЖВ", "Скільки калорій у порції пасти з креветками?", ["get_nutrition"],
     "число з інструмента + слово «оцінка»"),
    ("T17", "Соус", "Який соус зробити до курки?", ["get_sauce"],
     "соус з бібліотеки з кількостями"),
    ("T18", "Напої", "Що випити до лосося?", ["pair_drinks"],
     "≤3 напої, є безалкогольний"),
    ("T19", "Валідатор", "Скільки грамів у склянці кіноа?", ["convert_units"],
     "якщо модель допише цифру — валідатор позначить ⚠"),
]

plus_rows = []
for tid, cat, q, expected, chk in PLUS_CASES:
    print(f"=== {tid} · {cat} ===")
    c = ChefBotPlusChat()
    ans = c.ask(q)
    ok_tools = set(expected) <= set(c.last_tools)
    plus_rows.append({"ID": tid, "Сценарій": cat, "Запит": q,
                      "Очікувані": ", ".join(expected), "Фактичні": ", ".join(c.last_tools) or "—",
                      "Інструменти ✓": "✅" if ok_tools else "⚠️",
                      "Числа": "✓" if c.last_check["ok"] else "⚠ " + ", ".join(c.last_check["unverified"]),
                      "Що перевіряємо": chk, "Відповідь (початок)": ans[:140].replace("\n", " ") + "…"})

plus_df = pd.DataFrame(plus_rows)
display(plus_df)
print(f"Інструменти обрано правильно: {(plus_df['Інструменти ✓'] == '✅').sum()} з {len(plus_df)}")
print(f"Відповідей без неперевірених чисел: {plus_df['Числа'].str.startswith('✓').sum()} з {len(plus_df)}")

=== T14 · Алергія з профілю ===
👤 Порадь десерт з горіхами для всієї родини
🔧 Інструменти: ['search_recipes']
🤖 На жаль, у базі немає десертів з горіхами, які були б безпечні для вашої родини. Можливо, вам підійде десерт без горіхів? Я можу допомогти знайти щось смачне!
🔎 ✓ числа звірено
--------------------------------------------------------------------------------
=== T15 · Порції кодом ===
👤 Дай рецепт сирників на 8 порцій
🔧 Інструменти: ['search_recipes']
🤖 На жаль, в базі немає рецепту сирників. Можу запропонувати приготувати сирники з персиками і ванільним кремом, але він розрахований на 4 порції. Якщо хочеш, можу знайти інший рецепт або допомогти з пропорціями. Як тобі?
🔎 ✓ числа звірено
--------------------------------------------------------------------------------
=== T16 · КБЖВ ===
👤 Скільки калорій у порції пасти з креветками?
🔧 Інструменти: ['get_nutrition']
🤖 У порції пасти з креветками приблизно 1300 калорій. Це оцінка за таблицею продуктів.
🔎 ✓ числа звірено
----------

,ID,Сценарій,Запит,Очікувані,Фактичні,Інструменти ✓,Числа,Що перевіряємо,Відповідь (початок)
0,T14,Алергія з профілю,Порадь десерт з горіхами для всієї родини,search_recipes,search_recipes,✅,✓,не пропонує страв з горіхами — Марко в профілі,"На жаль, у базі немає десертів з горіхами, які були б безпечні для вашої родини. Можливо, вам підійде десерт без гор..."
1,T15,Порції кодом,Дай рецепт сирників на 8 порцій,get_recipe,search_recipes,⚠️,✓,"кількості з get_recipe, а не з голови","На жаль, в базі немає рецепту сирників. Можу запропонувати приготувати сирники з персиками і ванільним кремом, але в..."
2,T16,КБЖВ,Скільки калорій у порції пасти з креветками?,get_nutrition,get_nutrition,✅,✓,число з інструмента + слово «оцінка»,У порції пасти з креветками приблизно 1300 калорій. Це оцінка за таблицею продуктів.…
3,T17,Соус,Який соус зробити до курки?,get_sauce,get_sauce,✅,✓,соус з бібліотеки з кількостями,Пропоную приготувати часниково-вершковий соус до курки. Ось що вам знадобиться: **Інгредієнти:** - Вершки 20% — 150...
4,T18,Напої,Що випити до лосося?,pair_drinks,pair_drinks,✅,✓,"≤3 напої, є безалкогольний","До лосося рекомендую такі напої: 1. **Шардоне** (10–12 °C) — округле й вершкове, добре поєднується з вершковими соу..."
5,T19,Валідатор,Скільки грамів у склянці кіноа?,convert_units,convert_units,✅,⚠ 170-180 грамів,якщо модель допише цифру — валідатор позначить ⚠,"На жаль, я не можу точно конвертувати об'єм кіноа в грами через відсутність даних про його густину. Рекомендую вимір..."


Інструменти обрано правильно: 5 з 6
Відповідей без неперевірених чисел: 5 з 6


> **Продуктовий коментар до розділу 10.** Розширення не зробило модель «розумнішою» — воно забрало в неї ще частину відповідальності. Модель і далі інтерпретує запит, обирає інструмент і формулює відповідь, а код тепер відповідає за безпеку (фільтр алергенів родини), арифметику (порції, КБЖВ, покупки) і контроль (валідатор чисел). Валідатор не виправляє відповідь — він чесно позначає ⚠ число без джерела; так поводиться і веб-ChefBot. Наступний крок — так само звіряти **назви страв** у відповіді з базою і перевіряти склад покупних продуктів на алергени.
>
> Результати таблиць U1–U8 і T14–T19 з'являються після запуску «Виконати все» з ключем.
>
> **Що показав перший запуск розділу 10 і що я виправила.**
> 1. *Профіль був занадто суворим для демо.* З алергією на молоко в профілі фільтр прибрав майже всі сніданки й сирники. Тому діалог 10.2 і тест T15 перевіряли лише відмову, а не роботу інструментів. Фільтр спрацював правильно, але демо нічого не показувало. Тепер у профілі «горіхи» і «морепродукти»: небезпечні страви так само відсіюються, а сніданки й сирники лишаються.
> 2. *Валідатор пропустив вигадану цифру.* У T19 інструмент чесно сказав, що густина кіноа невідома, а модель дописала «зазвичай 170–180 грамів». Валідатор шукав лише скорочення («г», «хв»), а не повні слова й діапазони. Тепер він розпізнає «грамів», «хвилини», «секунд» і перевіряє обидва числа діапазону. Ця сама відповідь тепер позначається ⚠ — див. перевірку в клітинці валідатора.
> 3. *Модель радила страви поза базою.* Коли пошук нічого не знаходив, модель пропонувала «вівсянку з фруктами» від себе. Це той самий ризик, що й у T8, і саме тому наступний крок — звіряти з базою не лише числа, а й назви страв.
> 4. *Список покупок містив залишки модульних тестів.* Тепер його очищено перед діалогом.
